# Elastic 2-D/3-D Kirishima benchmark — rebuilt October 2026

Restarting flexOPT's Kirishima elastic-wave benchmark from scratch, step by step, checking each step before moving to the next:

1. Cartesian box around Kirishima in 2D/3D (this notebook), exactly as `SimuKirishima.ipynb` builds it.
2. No-free-surface benchmark against SPECFEM2D -- homogeneous and heterogeneous elastic, FD3/OPT3/OPT4/OPT5 -- **stop and review here**.
3. Flat free surface.
4. Real topography.

`src/` stays as given; anything this notebook needs that is not already in `src/` goes in `helpers/` instead, so `src/` does not need touching. The previous `Elastic2DtopoBenchmark_sep2026.ipynb` (Stage 1/2a/2b) is left untouched as a reference, not edited or deleted.

## Step 1 -- Cartesian box around Kirishima (2D/3D)

Reuses `SimuKirishima.ipynb`'s own construction primitives verbatim (`constructLocalBox`, `getParamsAndTopo`, the NIED/`planet1D` velocity model, the same cache names) for both the 3D box (for a later 3D elastic run) and the fast 2D x-z slice through the same centre (for the FD3/OPT3/OPT4/OPT5-vs-SPECFEM2D benchmark in the steps that follow). No free surface, Cerjan boundary or material-filling logic yet -- that is deliberately deferred to Steps 2-4 so this step stays pure geometry + velocity model.

In [ ]:
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        isfile(joinpath(candidate, "src", "flexOPT.jl")) && return candidate
    end
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)

# Only the geometry/velocity-model chain is needed for Step 1 -- no
# Metal/flexOPT/elasticWave2D yet (those arrive in Step 2 once we start
# propagating waves).
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints
using CairoMakie, Statistics, LinearAlgebra
CairoMakie.activate!(type="png")
@show VERSION Base.active_project()

In [ ]:
# Kirishima summit -- same point and same box/model construction as
# SimuKirishima.ipynb's "making an initial model" section.
kirishimaSummit = GeoPoint(dmsToDecimal(31, 56, 03), dmsToDecimal(130, 51, 42))
p0 = kirishimaSummit

# Keep the 3D model moderate; the 2D slice is resolved independently
# (its OPT recipe is nondimensional, so it is not rebuilt when these
# physical spacings change).
Δx3D = 200.0 # m
Δy3D = 200.0 # m
Δz3D = 200.0 # m
Δx2D = 100.0 # m
Δz2D = 100.0 # m
altMax = 2.e3 # m
altMin = -40.e3 # m
horizontalHalfWidth = 40.e3 # m; the model spans 80 km horizontally

boxGrids3D = constructLocalBox(
    p0, Δx3D, Δy3D, Δz3D,
    -horizontalHalfWidth, horizontalHalfWidth,
    -horizontalHalfWidth, horizontalHalfWidth,
    altMin, altMax,
)
boxGrids2D = constructLocalBox(
    p0, Δx2D, Δz2D,
    -horizontalHalfWidth, horizontalHalfWidth,
    altMin, altMax,
)

niedVelocitySource = DEFAULT_NIED_VELOCITY_SOURCE[]
velocitySourceKey = nied_velocity_source_key(niedVelocitySource)

gridCacheTag3D = "$(join(size(boxGrids3D.allGridsInGeoPoints), 'x'))_" *
    "d$(boxGrids3D.Δx)_$(boxGrids3D.Δy)_$(boxGrids3D.Δz)"
modelCacheName3D = "seismicModel3D_Kirishima_$(gridCacheTag3D)_NIED_$(velocitySourceKey)"
seismicModel3D = lazyProduceOrLoad(
    modelCacheName3D,
    getParamsAndTopo,
    boxGrids3D.allGridsInGeoPoints,
    boxGrids3D.effectiveRadii,
    0.1;
    velocity_model=:NIED,
    nied_source=niedVelocitySource,
    nied_confidence_max=0.8,
    nied_outside=:planet1D,
    nied_low_confidence=:planet1D,
)
@assert size(seismicModel3D.ρ) == size(boxGrids3D.allGridsInGeoPoints)

gridCacheTag2D = "$(join(size(boxGrids2D.allGridsInGeoPoints), 'x'))_" *
    "d$(boxGrids2D.Δx)_$(boxGrids2D.Δz)"
modelCacheName2D = "seismicModel2D_Kirishima_$(gridCacheTag2D)_NIED_$(velocitySourceKey)"
seismicModel2D = lazyProduceOrLoad(
    modelCacheName2D,
    getParamsAndTopo,
    boxGrids2D.allGridsInGeoPoints,
    boxGrids2D.effectiveRadii,
    0.1;
    velocity_model=:NIED,
    nied_source=niedVelocitySource,
    nied_confidence_max=0.8,
    nied_outside=:planet1D,
    nied_low_confidence=:planet1D,
)
@assert size(seismicModel2D.ρ) == size(boxGrids2D.allGridsInGeoPoints)

@show seismicModel3D.velocity_model seismicModel3D.nied_source
@show count(seismicModel3D.nied_mask) / length(seismicModel3D.nied_mask)
@show seismicModel2D.velocity_model seismicModel2D.nied_source

In [ ]:
# getParamsAndTopo uses ρAir=0.001 by default; 0.01 cleanly separates
# rock from air/void (same threshold as SimuKirishima.ipynb). No
# free-surface case, material-filling or boundary-condition logic yet --
# that is Steps 2-4.
air_density_cutoff = 0.01
material3D = (seismicModel3D.ρ .> air_density_cutoff) .& (seismicModel3D.Vsv .> 0)
material2D = (seismicModel2D.ρ .> air_density_cutoff) .& (seismicModel2D.Vsv .> 0)

# --- 2D topography (x, z in metres; what Steps 2-4 will build directly on) ---
xCoordinates2D = [p.xz[1] for p in boxGrids2D.allGridsInCartesian[:, 1]]
zCoordinates2D = [p.xz[2] for p in boxGrids2D.allGridsInCartesian[1, :]]
topographicSurfaceIndices2D = [
    findlast(@view material2D[ix, :]) for ix in axes(material2D, 1)]
@assert all(!isnothing, topographicSurfaceIndices2D) "a column is all air/void: widen altMax, or all solid: widen altMin"
topographicSurfaceIndices2D = Int.(topographicSurfaceIndices2D)
surfaceZ2D = [zCoordinates2D[k] for k in topographicSurfaceIndices2D]

# --- 3D topography (local x, y, z in km, matching SimuKirishima.ipynb's
#     own plotting convention) ---
x3DKm = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:, 1, 1]] .* 1e-3
y3DKm = [p.xyz[2] for p in boxGrids3D.allGridsInCartesian[1, :, 1]] .* 1e-3
z3DKm = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1, 1, :]] .* 1e-3
topography3D = Matrix{Float64}(undef, length(x3DKm), length(y3DKm))
for j in eachindex(y3DKm), i in eachindex(x3DKm)
    k = findlast(@view material3D[i, j, :])
    isnothing(k) && error("No material in column (i, j) = $((i, j))")
    k == length(z3DKm) && error("Topography reaches altMax in column (i, j) = $((i, j)); increase altMax")
    all(@view material3D[i, j, 1:k]) || error("Material mask has a cavity at $((i, j))")
    topography3D[i, j] = (z3DKm[k] + z3DKm[k + 1]) / 2
end

@show size(material2D) count(material2D) extrema(surfaceZ2D)
@show size(material3D) count(material3D) extrema(topography3D)
@show extrema(seismicModel2D.Vpv[material2D]) extrema(seismicModel2D.Vsv[material2D])

# 2D Vp/Vs cross-section, same as the figure the later FD3/OPT3/OPT4/OPT5
# steps will use to sanity-check their own heterogeneous model.
step1Figure2D = Figure(size=(1100, 420))
vpAxis2D = Axis(step1Figure2D[1, 1]; xlabel="x (km)", ylabel="z (km)",
    aspect=DataAspect(), title="Kirishima 2-D slice — Vp")
vsAxis2D = Axis(step1Figure2D[1, 2]; xlabel="x (km)", ylabel="z (km)",
    aspect=DataAspect(), title="Kirishima 2-D slice — Vs")
vpRange2D = extrema(seismicModel2D.Vpv[material2D])
vsRange2D = extrema(seismicModel2D.Vsv[material2D])
vpPlot2D = heatmap!(vpAxis2D, xCoordinates2D ./ 1e3, zCoordinates2D ./ 1e3,
    seismicModel2D.Vpv; colormap=:turbo, colorrange=vpRange2D)
vsPlot2D = heatmap!(vsAxis2D, xCoordinates2D ./ 1e3, zCoordinates2D ./ 1e3,
    seismicModel2D.Vsv; colormap=:turbo, colorrange=vsRange2D)
lines!(vpAxis2D, xCoordinates2D ./ 1e3, surfaceZ2D ./ 1e3; color=:white, linewidth=2)
lines!(vsAxis2D, xCoordinates2D ./ 1e3, surfaceZ2D ./ 1e3; color=:white, linewidth=2)
Colorbar(step1Figure2D[2, 1], vpPlot2D; vertical=false, label="Vp (km/s)")
Colorbar(step1Figure2D[2, 2], vsPlot2D; vertical=false, label="Vs (km/s)")
display(step1Figure2D)

# 3D shaded-relief check, same recipe as SimuKirishima.ipynb.
step1Figure3D = Figure(size=(1000, 800))
axis3D = Axis3(step1Figure3D[1, 1]; xlabel="local x (km)", ylabel="local y (km)",
    zlabel="elevation (km)", title="Kirishima 3-D box topography", aspect=:data)
terrain3D = surface!(axis3D, x3DKm, y3DKm, topography3D; color=topography3D,
    colormap=:terrain, colorrange=extrema(topography3D))
contour3d!(axis3D, x3DKm, y3DKm, topography3D;
    levels=range(extrema(topography3D)...; length=15), color=(:black, 0.35), linewidth=1)
Colorbar(step1Figure3D[1, 2], terrain3D; label="elevation (km)")
display(step1Figure3D)
nothing

## Step 2 -- no free surface, FD3 / OPT3 / OPT4 / OPT5 vs SPECFEM2D

Homogeneous and heterogeneous elastic cases, both embedded in the Kirishima structure from Step 1 but kept entirely below the real topography (no free surface yet, Cerjan absorbing layers on every side) -- the box is built fresh here (`step2-box-model`) rather than reusing Step 1's shallow box, since this benchmark needs to stay clear of the surface by construction. OPT4/OPT5 are enabled (the coefficient audit gate has passed) and the OPT propagation now exercises the predictor-corrector scheme (`:lumped_future`), matching the predictor discussed earlier -- the iteration/residual diagnostics print per case.

**Stop and review here before Step 3 (flat free surface).**


In [ ]:
using Metal
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
backend isa KernelAbstractions.CPU && @warn(
    "No GPU backend is available; OPT recipes will be built on CPU")
@show backend
include(joinpath(flexopt_root, "src", "elasticWave2D.jl"))
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .elasticWave2D, .flexOPT, .specfemBenchmark
using SparseArrays, Base64, JLD2


In [ ]:
# A Kirishima sub-volume that stays entirely below the real topography, so
# there is no free surface to model yet. Reuses `p0`/`niedVelocitySource`/
# `velocitySourceKey` from Step 1 as the geographic reference, but the box
# itself is centred ON THE SOURCE (offset step2SourceOffsetX from p0,
# depth step2CenterDepth), exactly like the existing
# Homogeneous/HeterogeneousElastic2DBenchmark notebooks -- that guarantees
# the full step2HalfWidth margin in every direction from the source to the
# absorbing boundary, which the receiver-grid safety assertion below needs.
# (A box centred on p0 instead left only ~15 km on the near side, since the
# source sits 10 km off-centre -- not enough margin at quickRun's 20 km
# threshold.)
referenceDx = 500.0        # same physical scale as the existing benchmarks
fdOptSpatialRefinement = 2 # dx = 250 m, the proven stable FD3/OPT3 setup
dx2D = referenceDx / fdOptSpatialRefinement
step2HalfWidth = 50.e3     # enlarged from 45 km: the real Kirishima vpMaximum
                           # (possibly uppermost-mantle, not just crustal) needs
                           # more margin to clear the edge-arrival assertion below
step2CenterDepth = -55.e3  # keeps the shallow bound at -5 km (step2CenterDepth +
                           # step2HalfWidth == -5e3), same safety margin as before
step2SourceOffsetX = -10.e3 # same source offset used by the existing benchmarks
boxGrids2DStep2 = constructLocalBox(
    p0, dx2D, dx2D,
    step2SourceOffsetX - step2HalfWidth, step2SourceOffsetX + step2HalfWidth,
    step2CenterDepth - step2HalfWidth, step2CenterDepth + step2HalfWidth,
)
gridCacheTagStep2 = "$(join(size(boxGrids2DStep2.allGridsInGeoPoints), 'x'))_" *
    "d$(boxGrids2DStep2.Δx)_$(boxGrids2DStep2.Δz)"
modelCacheNameStep2 = "seismicModel2DStep2_Kirishima_$(gridCacheTagStep2)_NIED_$(velocitySourceKey)"
seismicModel2DStep2 = lazyProduceOrLoad(
    modelCacheNameStep2, getParamsAndTopo,
    boxGrids2DStep2.allGridsInGeoPoints, boxGrids2DStep2.effectiveRadii, 0.1;
    velocity_model=:NIED, nied_source=niedVelocitySource,
    nied_confidence_max=0.8, nied_outside=:planet1D, nied_low_confidence=:planet1D,
)
@assert size(seismicModel2DStep2.ρ) == size(boxGrids2DStep2.allGridsInGeoPoints)
# Sanity check: the whole box must already be solid material (no void/air),
# since Step 2 has no free surface by construction, not by masking.
@assert all(seismicModel2DStep2.ρ .> air_density_cutoff) "step2CenterDepth/step2HalfWidth let the box reach the surface -- push it deeper"
xCoordinatesStep2 = [p.xz[1] for p in boxGrids2DStep2.allGridsInCartesian[:, 1]]
zCoordinatesStep2 = [p.xz[2] for p in boxGrids2DStep2.allGridsInCartesian[1, :]]
nxStep2, nzStep2 = length(xCoordinatesStep2), length(zCoordinatesStep2)
sourcePositionStep2 = (x=step2SourceOffsetX, z=step2CenterDepth)
@show size(seismicModel2DStep2.ρ) extrema(seismicModel2DStep2.Vpv) extrema(seismicModel2DStep2.Vsv)
@show nxStep2 nzStep2 dx2D step2HalfWidth step2CenterDepth sourcePositionStep2


In [ ]:
quickRun = true
optSpatialRefinement = 1   # FD3, convFD3 and OPT3 share dx (no OPT refinement yet)
buildHigherOrderRecipes = true
runOPT4 = false # 4-point stencil is inherently asymmetric -- dropped, nothing to fix
runOPT5 = true  # ElasticLHSOperatorAudit's coefficient gate has already passed
runOPTWithoutGamma = true
runConvFDBeyondBoundarySafeTime = false
waveformQuantity = :displacement
runSPECFEM2D = true
temporalRefinement = 2
optTimeMarchingScheme = :predictor_corrector # exercise the P-C scheme now
optPredictorMode = :lumped_future
optCorrectorIterations = 3
optCorrectorTolerance = 1e-8
optCorrectorRelaxation = 1.0

dx = dx2D
x = xCoordinatesStep2
z = zCoordinatesStep2
nx, nz = length(x), length(z)
sourcePosition = sourcePositionStep2
applyFreeSurface = false
applyCerjan = !quickRun
material = trues(nx, nz)
vp0, vs0, rho0 = (
    median(seismicModel2DStep2.Vpv) * 1e3,
    median(seismicModel2DStep2.Vsv) * 1e3,
    median(seismicModel2DStep2.ρ) * 1e3,
)
cerjanCells = round(Int, 24 * referenceDx / dx)
cerjan = CerjanBoundarySpec(
    (cerjanCells, cerjanCells),
    (cerjanCells, cerjanCells); # no free surface: absorb on every side
    damping=0.0053 / fdOptSpatialRefinement^2,
)
activeCerjan = applyCerjan ? cerjan : nothing
bcFD = BoundaryConditionSet(free_surface=nothing, cerjan=activeCerjan,
    material_mask=BitArray(material), free_surface_mode=:pinned_void)
receiverZ = sourcePosition.z + 5e3
receiverX = quickRun ?
    (sourcePosition.x .+ [-20e3, -15e3, -10e3, -5e3, 0.0]) :
    (sourcePosition.x .+ [-20e3, -10e3, 0.0, 10e3, 20e3])
receiverGridX = quickRun ? collect((sourcePosition.x-20e3):5e3:(sourcePosition.x)) :
    collect((sourcePosition.x-30e3):10e3:(sourcePosition.x+30e3))
receiverGridZ = quickRun ? collect((sourcePosition.z-15e3):5e3:(sourcePosition.z)) :
    collect((sourcePosition.z-30e3):10e3:(sourcePosition.z+10e3))
receiverGrid = [(x=xr, z=zr) for zr in receiverGridZ for xr in receiverGridX
    if hypot(xr - sourcePosition.x, zr - sourcePosition.z) >= 5e3]
receiverBoundaryMargins = [minimum((
    station.x - first(x), last(x) - station.x,
    station.z - first(z), last(z) - station.z,
)) for station in receiverGrid]
@assert minimum(receiverBoundaryMargins) >= (quickRun ? 20e3 : 30e3)
duration = quickRun ? 6.4 : 12.0
outputSampling = 0.10
referencePointsPerSWavelength = 16
rickerSharpness = 1.5
minimumAcceptedPointsPerSWavelength = 10
coarsestSpacing = referenceDx
sourceFrequency = rickerSharpness * vs0 /
    (referencePointsPerSWavelength * coarsestSpacing)
sourceDelay = 1.5 / sourceFrequency
sourceForce = 1.0e10
sourceMode = :gaussian
sourceSpatialSigma = referenceDx
sourceCutoffSigmas = 2.0
function spatial_source_specs(xaxis, zaxis; center=sourcePosition,
    mode=sourceMode, sigma=sourceSpatialSigma, cutoff=sourceCutoffSigmas)
    if mode === :point
        return [(x=xaxis[argmin(abs.(xaxis .- center.x))],
            z=zaxis[argmin(abs.(zaxis .- center.z))], weight=1.0)]
    end
    specs = [(x=Float64(xv), z=Float64(zv),
        weight=exp(-((xv-center.x)^2 + (zv-center.z)^2) / (2sigma^2)))
        for xv in xaxis for zv in zaxis
        if hypot(xv-center.x, zv-center.z) <= cutoff*sigma]
    total = sum(s.weight for s in specs)
    [(x=s.x, z=s.z, weight=s.weight / total) for s in specs]
end
sourceSpecsReference = spatial_source_specs(x, z)
@assert isapprox(sum(s.weight for s in sourceSpecsReference), 1.0; atol=1e-14)
rickerSource(time) = begin
    a = π * sourceFrequency * (time - sourceDelay)
    (1 - 2a^2) * exp(-a^2)
end
distanceToNearestPhysicalEdge = minimum((
    sourcePosition.x - first(x), last(x) - sourcePosition.x,
    sourcePosition.z - first(z), last(z) - sourcePosition.z,
))
significantSourceStart = sourceDelay - 1 / sourceFrequency
@show applyFreeSurface applyCerjan (nx, nz) sourcePosition receiverZ dx
@show sourceFrequency sourceDelay rickerSharpness sourceForce sourceMode
@show vp0 vs0 rho0 cerjanCells distanceToNearestPhysicalEdge
receiverGeometryFigure = Figure(size=(720, 620))
receiverGeometryAxis = Axis(receiverGeometryFigure[1, 1];
    xlabel="x (km)", ylabel="z (km)", aspect=DataAspect(),
    title="Step 2 receiver grid around the Kirishima-structure source")
scatter!(receiverGeometryAxis, getproperty.(receiverGrid, :x) ./ 1e3,
    getproperty.(receiverGrid, :z) ./ 1e3; label="receiver grid")
scatter!(receiverGeometryAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
    marker=:star5, markersize=20, color=:gold, strokecolor=:black, label="source")
axislegend(receiverGeometryAxis)
display(receiverGeometryFigure)


### Homogeneous sub-case


In [ ]:
# Homogeneous sub-case: a single representative medium (the box median),
# so FD3/OPT3/OPT4/OPT5 are compared against SPECFEM2D exactly as in
# HomogeneousElastic2DBenchmark.ipynb, but using Kirishima-scale constants.
vpFieldHomog = fill(vp0, nx, nz)
vsFieldHomog = fill(vs0, nx, nz)
rhoFieldHomog = fill(rho0, nx, nz)
modelHomog = (ρ=rhoFieldHomog ./ 1e3, Vpv=vpFieldHomog ./ 1e3, Vsv=vsFieldHomog ./ 1e3)
vpMaximumHomog, vsMinimumHomog = maximum(vpFieldHomog), minimum(vsFieldHomog)
pointsPerSWavelengthHomog = (
    FD3=vsMinimumHomog / (sourceFrequency * dx),
    OPT3=vsMinimumHomog / (sourceFrequency * (dx / optSpatialRefinement)),
    OPT5=vsMinimumHomog / (sourceFrequency * (dx / optSpatialRefinement)),
)
earliestPEdgeArrivalHomog = significantSourceStart +
    distanceToNearestPhysicalEdge / vpMaximumHomog
@assert duration < earliestPEdgeArrivalHomog
@assert minimum(values(pointsPerSWavelengthHomog)) >=
    minimumAcceptedPointsPerSWavelength
@show vpMaximumHomog vsMinimumHomog pointsPerSWavelengthHomog earliestPEdgeArrivalHomog


In [ ]:
dxOPTHomog = dx / optSpatialRefinement
xOPTPhysicalHomog = collect(first(x):dxOPTHomog:last(x))
zOPTPhysicalHomog = collect(first(z):dxOPTHomog:last(z))
solidOPTHomog = applyFreeSurface ?
    repeat(reshape(zOPTPhysicalHomog .<= 0.0, 1, :),
        length(xOPTPhysicalHomog), 1) :
    trues(length(xOPTPhysicalHomog), length(zOPTPhysicalHomog))
dtOPTHomog = 0.20 * dxOPTHomog / (sqrt(2) * vpMaximumHomog * temporalRefinement)
@assert optSpatialRefinement == 1 "OPT refinement != 1 needs field resampling, not implemented for the Kirishima case yet"
vpFieldOPTHomog, vsFieldOPTHomog, rhoFieldOPTHomog = vpFieldHomog, vsFieldHomog, rhoFieldHomog
# Stable referenceHomog used by the previously working OPT benchmark. This is
# an exact change of coordinates x'=x/Δx, z'=z/Δz, t'=t/Δt, not a
# change of the physical modelHomog. The direct physical-recipe path remains
# available for conditioning experiments, but is not the benchmark default.
optFormulationHomog = :scaled_coordinates # or :physical_recipe_experimental
if optFormulationHomog === :scaled_coordinates
    recipeSpacingHomog = (1.0, 1.0, 1.0)
    rhoOPTHomog = rhoFieldOPTHomog ./ rho0
    muOPTHomog = (rhoFieldOPTHomog .* vsFieldOPTHomog.^2 ./ rho0) .* (dtOPTHomog/dxOPTHomog)^2
    lambdaOPTHomog = (rhoFieldOPTHomog .* (vpFieldOPTHomog.^2 .- 2vsFieldOPTHomog.^2) ./ rho0) .*
        (dtOPTHomog/dxOPTHomog)^2
elseif optFormulationHomog === :physical_recipe_experimental
    recipeSpacingHomog = (dxOPTHomog, dxOPTHomog, dtOPTHomog)
    rhoOPTHomog = rhoFieldOPTHomog
    muOPTHomog = rhoFieldOPTHomog .* vsFieldOPTHomog.^2
    lambdaOPTHomog = rhoFieldOPTHomog .* (vpFieldOPTHomog.^2 .- 2vsFieldOPTHomog.^2)
else
    error("unknown optFormulation=$optFormulationHomog")
end
muOPTHomog[.!solidOPTHomog] .= 0.0
lambdaOPTHomog[.!solidOPTHomog] .= 0.0
optParametersHomog = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
    "Δ" => recipeSpacingHomog,
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    # Preserve all total-degree <= 2 Taylor conditions exactly; use
    # supplementaryOrder only to optimise the remaining null space.
    "taylorInverseMode" => :hierarchical_constrained,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    # The expensive semi-symbolic coefficientHomog construction runs on the
    # GPU selected in the bootstrap cell (Metal here).
    "recipe_backend" => backend,
)
# Cache complete semi-symbolic recipes, not the very large prepared
# operators. The runtime Metal/CPU object is deliberately excluded from
# the cache key and is injected only when a recipe must be produced.
recipeCacheDirectoryHomog = "semiSymbolic"
recipeCacheVersionHomog = 3 # hierarchical Taylor inverse; reject legacy recipes
function cachedOPTRecipeHomog(parameters, prefix; existing=nothing)
    cacheParametersHomog = Dict{String,Any}(
        key => value for (key, value) in parameters
        if key != "recipe_backend"
    )
    cacheParametersHomog["recipe_cache_version"] = recipeCacheVersionHomog
    function produceRecipeHomog(config)
        runtimeParametersHomog = Dict{String,Any}(config)
        pop!(runtimeParametersHomog, "hash_id", nothing)
        pop!(runtimeParametersHomog, "recipe_cache_version", nothing)
        runtimeParametersHomog["recipe_backend"] = backend
        isnothing(existing) ?
            makeOPTsemiSymbolic(runtimeParametersHomog) : existing
    end
    myProduceOrLoad(produceRecipeHomog, cacheParametersHomog,
        recipeCacheDirectoryHomog, prefix)
end
existingOPT3RecipeHomog = @isdefined(optRecipeHomog) ? optRecipeHomog : nothing
optRecipeHomog = cachedOPTRecipeHomog(optParametersHomog, "elastic2D_OPT3";
    existing=existingOPT3RecipeHomog)
# P-C predictor: same three-point support and physical modelHomog, but only
# the supplementary Taylor optimisation is removed. The fullHomog OPT3
# operator remains the residual operator used by every correction.
optPredictorParametersHomog = copy(optParametersHomog)
optPredictorParametersHomog["supplementaryOrder"] = 0
optPredictorRecipeHomog = optTimeMarchingScheme === :predictor_corrector &&
    optPredictorMode === :prepared ?
    cachedOPTRecipeHomog(optPredictorParametersHomog, "elastic2D_OPT3_predictor_s0") : nothing

# Four/five-point spatial recipes; time remains a three-point stencil.
opt4ParametersHomog = copy(optParametersHomog)
opt4ParametersHomog["pointsInSpace"] = 4
opt4ParametersHomog["fieldItpl"] = merge(optParametersHomog["fieldItpl"],
    (offsetSpace=1.5,))
opt4ParametersHomog["materItpl"] = merge(optParametersHomog["materItpl"],
    (offsetSpace=1.5,))
opt4RecipeHomog = buildHigherOrderRecipes ? cachedOPTRecipeHomog(
    opt4ParametersHomog, "elastic2D_OPT4"; existing=nothing) : nothing

# Five-point OPT recipe: the time stencil remains three points so that
# only the spatial approximation changes relative to OPT3.
opt5ParametersHomog = copy(optParametersHomog)
# Validated OPT5 family used by seismo1Dbenchmark: five trial points
# retain the linear B-spline test order. orderBspace=3 is not OPT5 and
# generated a nearly singular first-step solve in this elastic problem.
opt5ParametersHomog["orderBspace"] = 1
opt5ParametersHomog["pointsInSpace"] = 5
opt5ParametersHomog["fieldItpl"] = merge(optParametersHomog["fieldItpl"],
    (offsetSpace=2.0,))
opt5ParametersHomog["materItpl"] = merge(optParametersHomog["materItpl"],
    (offsetSpace=2.0,))
# Keep OPT5 available, but do not even load/build it in the quick run.
# When enabled, never migrate the former invalid orderBspace=3 object.
opt5RecipeHomog = buildHigherOrderRecipes ? cachedOPTRecipeHomog(
    opt5ParametersHomog, "elastic2D_OPT5"; existing=nothing) : nothing

# Collocation/delta-test recipe corresponding to the conventional FD3
# definition used in seismo1Dbenchmark.ipynb. Keep it beside OPT3: it is
# a coefficientHomog referenceHomog, not an alias for the explicit flux solver.
conventionalFDParametersHomog = copy(optParametersHomog)
conventionalFDParametersHomog["orderBspace"] = -1
conventionalFDParametersHomog["orderBtime"] = -1
conventionalFDParametersHomog["supplementaryOrder"] = 0
existingConvFD3RecipeHomog = @isdefined(conventionalFDRecipeHomog) ?
    conventionalFDRecipeHomog : nothing
conventionalFDRecipeHomog = cachedOPTRecipeHomog(
    conventionalFDParametersHomog, "elastic2D_convFD3";
    existing=existingConvFD3RecipeHomog)
recipeComparisonHomog = (
    conventionalFD3=(orderBspace=-1, orderBtime=-1,
        supplementaryOrder=0, pointsInSpace=3, pointsInTime=3),
    OPT3=(orderBspace=optParametersHomog["orderBspace"],
        orderBtime=optParametersHomog["orderBtime"],
        supplementaryOrder=optParametersHomog["supplementaryOrder"],
        pointsInSpace=optParametersHomog["pointsInSpace"],
        pointsInTime=optParametersHomog["pointsInTime"]),
    OPT4=(orderBspace=opt4ParametersHomog["orderBspace"],
        orderBtime=opt4ParametersHomog["orderBtime"],
        supplementaryOrder=opt4ParametersHomog["supplementaryOrder"],
        pointsInSpace=opt4ParametersHomog["pointsInSpace"],
        pointsInTime=opt4ParametersHomog["pointsInTime"]),
    OPT5=(orderBspace=opt5ParametersHomog["orderBspace"],
        orderBtime=opt5ParametersHomog["orderBtime"],
        supplementaryOrder=opt5ParametersHomog["supplementaryOrder"],
        pointsInSpace=opt5ParametersHomog["pointsInSpace"],
        pointsInTime=opt5ParametersHomog["pointsInTime"]),
)
# Public handles for inspecting the complete symbolic recipes in IJulia,
# e.g. `optRecipesHomog.OPT5["recette"]`.
optRecipesHomog = (convFD3=conventionalFDRecipeHomog, OPT3=optRecipeHomog,
    OPT4=opt4RecipeHomog, OPT5=opt5RecipeHomog)
@show recipeComparisonHomog
if runOPT4 || runOPT5
    coefficientGatePathHomog = joinpath(flexopt_root, "data",
        "elastic_lhs_coefficient_gate.jld2")
    isfile(coefficientGatePathHomog) || error(
        "Run ElasticLHSOperatorAudit.ipynb before OPT4/OPT5 propagation")
    coefficientGateHomog = load(coefficientGatePathHomog)
    @assert all(result.passed for result in
        values(coefficientGateHomog["coefficient_health"]))
end



In [ ]:
optCerjanHomog = CerjanBoundarySpec(
    (cerjan.lower[1] * optSpatialRefinement,
        cerjan.lower[2] * optSpatialRefinement),
    (cerjan.upper[1] * optSpatialRefinement, 0);
    damping=cerjan.damping / optSpatialRefinement^2,
)
activeOptCerjanHomog = applyCerjan ? optCerjanHomog : nothing
bcOPTHomog = applyFreeSurface ?
    boundary_geometry(solidOPTHomog, (dxOPTHomog, dxOPTHomog);
        free_surface_mode=:pinned_void, cerjan=activeOptCerjanHomog) :
    BoundaryConditionSet(free_surface=nothing, cerjan=activeOptCerjanHomog,
        material_mask=BitArray(solidOPTHomog), free_surface_mode=:pinned_void)
modelsOPTHomog = [rhoOPTHomog, lambdaOPTHomog, muOPTHomog]
pointsOPTHomog = getModelPoints(modelsOPTHomog[1], 3,
    optRecipeHomog["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
familyOPTHomog = (models=modelsOPTHomog, modelPoints=pointsOPTHomog,
    Δ=recipeSpacingHomog, modelName="homogeneous_OPT3_$(optFormulationHomog)")
operatorTimingsHomog = Dict{Symbol,Float64}()
timedOPT3AssemblyHomog = @timed numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => optRecipeHomog, "modelFam" => familyOPTHomog,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPTHomog,
    "representation" => "matrixfree",
))["numOperators"]
numericalVolumeHomog = timedOPT3AssemblyHomog.value
operatorTimingsHomog[:OPT3_assembly] = timedOPT3AssemblyHomog.time
timedOPT3PreparationHomog = @timed prepareLinearSystem(numericalVolumeHomog;
    free_surface_spacing=recipeSpacingHomog[1:2])
preparedVolumeHomog = timedOPT3PreparationHomog.value
operatorTimingsHomog[:OPT3_preparation] = timedOPT3PreparationHomog.time
preparedOPT4VolumeHomog = if runOPT4
    pointsOPT4Homog = getModelPoints(modelsOPTHomog[1], 4,
        opt4RecipeHomog["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT4Homog = (models=modelsOPTHomog, modelPoints=pointsOPT4Homog,
        Δ=recipeSpacingHomog, modelName="homogeneous_OPT4_$(optFormulationHomog)")
    numericalOPT4Homog = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt4RecipeHomog, "modelFam" => familyOPT4Homog,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPTHomog, "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numericalOPT4Homog; free_surface_spacing=recipeSpacingHomog[1:2])
else
    nothing
end
preparedOPT5VolumeHomog = if runOPT5
    pointsOPT5Homog = getModelPoints(modelsOPTHomog[1], 5,
        opt5RecipeHomog["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT5Homog = (models=modelsOPTHomog, modelPoints=pointsOPT5Homog,
        Δ=recipeSpacingHomog, modelName="homogeneous_OPT5_$(optFormulationHomog)")
    numericalOPT5Homog = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt5RecipeHomog, "modelFam" => familyOPT5Homog,
        "absorbingBoundaries" => nothing,
        "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPTHomog,
        "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numericalOPT5Homog;
        free_surface_spacing=recipeSpacingHomog[1:2])
else
    nothing
end
timedConvFDAssemblyHomog = @timed numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => conventionalFDRecipeHomog, "modelFam" => familyOPTHomog,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPTHomog,
    "representation" => "matrixfree",
))["numOperators"]
numericalConvFDHomog = timedConvFDAssemblyHomog.value
operatorTimingsHomog[:convFD3_assembly] = timedConvFDAssemblyHomog.time
timedConvFDPreparationHomog = @timed prepareLinearSystem(numericalConvFDHomog;
    free_surface_spacing=recipeSpacingHomog[1:2])
preparedConvFDVolumeHomog = timedConvFDPreparationHomog.value
operatorTimingsHomog[:convFD3_preparation] = timedConvFDPreparationHomog.time

# Optional traction recipe σn=0, assembled additively on surface rowsHomog.
if applyFreeSurface
boundaryParametersHomog = copy(optParametersHomog)
boundaryParametersHomog["famousEquationType"] = "elasticTractionFree2D"
boundaryRecipeHomog = cachedOPTRecipeHomog(
    boundaryParametersHomog, "elasticTractionFree2D_OPT3")
normalXHomog = zeros(size(solidOPTHomog)); normalZ = zeros(size(solidOPTHomog))
for (point, normal) in zip(bcOPTHomog.free_surface.points, bcOPTHomog.free_surface.normals)
    normalXHomog[point], normalZ[point] = normal
end
boundaryModelsHomog = [lambdaOPTHomog, muOPTHomog, normalXHomog, normalZ]
boundaryPointsHomog = getModelPoints(boundaryModelsHomog[1], 3,
    boundaryRecipeHomog["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
boundaryFamilyHomog = (models=boundaryModelsHomog, modelPoints=boundaryPointsHomog,
    Δ=recipeSpacingHomog, modelName="homogeneous_free_surface")
numericalBoundaryHomog = numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => boundaryRecipeHomog, "modelFam" => boundaryFamilyHomog,
    "absorbingBoundaries" => (isnothing(bcOPTHomog.cerjan) ? nothing : cerjan_padding(bcOPTHomog.cerjan)),
    "maskedRegionInSpace" => bcOPTHomog.free_surface.points,
    "representation" => "matrixfree",
))["numOperators"]
preparedBoundaryHomog = prepareLinearSystem(numericalBoundaryHomog)
surfaceWholeHomog = numericalVolumeHomog.numericalOperators.left.geometry.freeSurfaceBoundary.points
    preparedOPTHomog = overlapBoundaryLinearSystem(
    preparedVolumeHomog, preparedBoundaryHomog, surfaceWholeHomog;
    mode=:additive, boundary_weight=1.0,
    )
    preparedConvFDHomog = overlapBoundaryLinearSystem(
        preparedConvFDVolumeHomog, preparedBoundaryHomog, surfaceWholeHomog;
        mode=:additive, boundary_weight=1.0,
    )
    preparedOPT4Homog = runOPT4 ? overlapBoundaryLinearSystem(
        preparedOPT4VolumeHomog, preparedBoundaryHomog, surfaceWholeHomog;
        mode=:additive, boundary_weight=1.0) : nothing
    preparedOPT5Homog = runOPT5 ? overlapBoundaryLinearSystem(
        preparedOPT5VolumeHomog, preparedBoundaryHomog, surfaceWholeHomog;
        mode=:additive, boundary_weight=1.0) : nothing
else
    preparedOPTHomog = preparedVolumeHomog
    preparedConvFDHomog = preparedConvFDVolumeHomog
    preparedOPT4Homog = runOPT4 ? preparedOPT4VolumeHomog : nothing
    preparedOPT5Homog = runOPT5 ? preparedOPT5VolumeHomog : nothing
end
@assert preparedOPTHomog.NForceField == 2
@assert preparedConvFDHomog.NForceField == 2
runOPT4 && @assert preparedOPT4Homog.NForceField == 2
runOPT5 && @assert preparedOPT5Homog.NForceField == 2
optOperatorUnitConventionHomog = optFormulationHomog
boundaryOverlapRowsHomog = hasproperty(preparedOPTHomog, :boundary_overlap_rows) ?
    preparedOPTHomog.boundary_overlap_rows : Int[]
@show quickRun optFormulationHomog optSpatialRefinement dxOPTHomog dtOPTHomog recipeSpacingHomog
@show operatorTimingsHomog
@show preparedOPTHomog.spaceShape preparedConvFDHomog.spaceShape runOPT5 length(boundaryOverlapRowsHomog)
runOPT5 && @show preparedOPT5Homog.spaceShape


In [ ]:
optPaddingHomog = isnothing(bcOPTHomog.cerjan) ? zeros(Int, 2, 2) : cerjan_padding(bcOPTHomog.cerjan)
@assert optOperatorUnitConventionHomog === optFormulationHomog "restart the kernel and rerun the OPT operator cell"
xOPTHomog = range(first(xOPTPhysicalHomog) - optPaddingHomog[1,1] * dxOPTHomog;
    step=dxOPTHomog, length=preparedOPTHomog.spaceShape[1])
zOPTHomog = range(first(zOPTPhysicalHomog) - optPaddingHomog[1,2] * dxOPTHomog;
    step=dxOPTHomog, length=preparedOPTHomog.spaceShape[2])
sourceIXOPTHomog = argmin(abs.(xOPTPhysicalHomog .- sourcePosition.x))
sourceIZOPTHomog = argmin(abs.(zOPTPhysicalHomog .- sourcePosition.z))
sourceIndexOPTHomog = CartesianIndex(
    sourceIXOPTHomog + optPaddingHomog[1,1], sourceIZOPTHomog + optPaddingHomog[1,2])
sourceLinearOPTHomog = LinearIndices(preparedOPTHomog.spaceShape)[sourceIndexOPTHomog]
sourceSpecsOPTHomog = spatial_source_specs(xOPTPhysicalHomog, zOPTPhysicalHomog)
sourceNodesOPTHomog = [(linear=LinearIndices(preparedOPTHomog.spaceShape)[
        CartesianIndex(argmin(abs.(xOPTHomog .- s.x)),
            argmin(abs.(zOPTHomog .- s.z)))], weight=s.weight)
    for s in sourceSpecsOPTHomog]
@assert isapprox(sum(s.weight for s in sourceNodesOPTHomog), 1.0; atol=1e-14)
optStepsHomog = ceil(Int, duration / dtOPTHomog)
optOutputStrideHomog = max(1, round(Int, outputSampling / dtOPTHomog))
# Align the force stencil with (past..., future): for three time points,
# the first solve for u(+Δt) receives f(-Δt), f(0), f(+Δt).
nPastForceLevelsHomog = preparedOPTHomog.timePointsUsedForOneStep - 1
sourceTimesOPTHomog = ((1 - nPastForceLevelsHomog):optStepsHomog) .* dtOPTHomog
@assert sourceTimesOPTHomog[preparedOPTHomog.timePointsUsedForOneStep] == dtOPTHomog
waveletOPTHomog = rickerSource.(sourceTimesOPTHomog)
# In SI, famousEquations uses f=ρa and the cell body-force density is
# sourceForce/(ΔxΔz). Under the scaled-coordinate PDE this becomes
# f'=f Δt²/ρ = sourceForce Δt²/(ρ ΔxΔz).
sourceScaleOPTHomog = optFormulationHomog === :scaled_coordinates ?
    sourceForce * dtOPTHomog^2 / (rho0 * dxOPTHomog^2) :
    sourceForce / dxOPTHomog^2
@assert optFormulationHomog !== :scaled_coordinates ||
    isapprox(sourceScaleOPTHomog * rho0 * dxOPTHomog^2 / dtOPTHomog^2, sourceForce)
sourceOPTHomog = zeros(Float64, preparedOPTHomog.NforcePoints,
    preparedOPTHomog.NForceField, length(sourceTimesOPTHomog))
for sourceNode in sourceNodesOPTHomog
    sourceOPTHomog[sourceNode.linear, 2, :] .+=
        sourceNode.weight .* sourceScaleOPTHomog .* waveletOPTHomog
end
predictorSelectionHomog = optPredictorMode === :lumped_future ?
    (predictor_mode=:lumped_future,) :
    (predictor_mode=:prepared, predictor_prepared=preparedPredictorVolume)
optMarchingOptionsHomog = optTimeMarchingScheme === :direct ?
    (scheme=:direct,) : merge((
        scheme=:predictor_corrector,
        corrector_iterations=optCorrectorIterations,
        corrector_tolerance=optCorrectorTolerance,
        corrector_relaxation=optCorrectorRelaxation,
    ), predictorSelectionHomog)
propagationOPTHomog = propagateLinearSystem(
    preparedOPTHomog, optStepsHomog, dtOPTHomog;
    sourceFull=sourceOPTHomog, output_stride=optOutputStrideHomog,
    blowup_limit=1e8,
    solver_name="OPT3",
    optMarchingOptionsHomog...,
)
@assert !propagationOPTHomog.stopped_early
if !isnothing(propagationOPTHomog.predictor_corrector)
    pcHomog = propagationOPTHomog.predictor_corrector
    @show extrema(pcHomog.iterations_per_step) extrema(pcHomog.relative_residuals)
    @show propagationOPTHomog.timing
end

# Controlled Γ diagnostic. Preserve, for every temporal source slot, the
# signed columnHomog sum of R_force but place it only on the collocated equation
# row. Thus only the spatial redistribution is removed.
function collocated_source_operatorHomog(prepared, source_nodes; force_field=2)
    rowsHomog=Int[]; columns=Int[]; values=Float64[]
    nspaceHomog, nfieldHomog = prepared.NpointsSpace, prepared.NField
    for node in source_nodes, time_slot in 1:prepared.timePointsUsedForOneStep
        columnHomog = node.linear + (force_field-1)*nspaceHomog +
            (time_slot-1)*nspaceHomog*prepared.NForceField
        coefficientHomog = sum(@view prepared.R_force[:, columnHomog])
        push!(rowsHomog, force_field + nfieldHomog*(node.linear-1))
        push!(columns, columnHomog); push!(values, coefficientHomog)
    end
    sparse(rowsHomog, columns, values, size(prepared.R_force)...)
end
RForceCollocatedHomog = collocated_source_operatorHomog(preparedOPTHomog, sourceNodesOPTHomog)
preparedOPTWithoutGammaHomog = runOPTWithoutGamma ?
    flexOPT._prepared_with_matrices(preparedOPTHomog, preparedOPTHomog.A_unknown,
        preparedOPTHomog.L_known, RForceCollocatedHomog; source_mode=:collocated) : nothing
propagationOPTWithoutGammaHomog = runOPTWithoutGamma ? propagateLinearSystem(
    preparedOPTWithoutGammaHomog, optStepsHomog, dtOPTHomog; sourceFull=sourceOPTHomog,
    output_stride=optOutputStrideHomog, blowup_limit=1e8,
    solver_name="OPT3 without Γ redistribution",
) : nothing
runOPTWithoutGamma && @assert !propagationOPTWithoutGammaHomog.stopped_early
uxOPTHomog = propagationOPTHomog.history[:, :, 1, :]
uzOPTHomog = propagationOPTHomog.history[:, :, 2, :]
optTimesHomog = propagationOPTHomog.times
uxOPTWithoutGammaHomog = runOPTWithoutGamma ?
    propagationOPTWithoutGammaHomog.history[:, :, 1, :] : nothing
uzOPTWithoutGammaHomog = runOPTWithoutGamma ?
    propagationOPTWithoutGammaHomog.history[:, :, 2, :] : nothing

# Higher-order runs are gated by ElasticLHSOperatorAudit.ipynb.
if runOPT4
    sourceOPT4Homog = zeros(Float64, preparedOPT4Homog.NforcePoints,
        preparedOPT4Homog.NForceField, length(sourceTimesOPTHomog))
    for sourceNode in sourceNodesOPTHomog
        sourceOPT4Homog[sourceNode.linear, 2, :] .+=
            sourceNode.weight .* sourceScaleOPTHomog .* waveletOPTHomog
    end
    propagationOPT4Homog = propagateLinearSystem(preparedOPT4Homog, optStepsHomog, dtOPTHomog;
        sourceFull=sourceOPT4Homog, output_stride=optOutputStrideHomog,
        blowup_limit=1e8, solver_name="OPT4")
    @assert !propagationOPT4Homog.stopped_early
    uxOPT4Homog = propagationOPT4Homog.history[:, :, 1, :]
    uzOPT4Homog = propagationOPT4Homog.history[:, :, 2, :]
    opt4TimesHomog = propagationOPT4Homog.times
else
    propagationOPT4Homog = uxOPT4Homog = uzOPT4Homog = opt4TimesHomog = nothing
end

# OPT5 remains available, but is skipped in the quick benchmark by default.
if runOPT5
    sourceOPT5Homog = zeros(Float64, preparedOPT5Homog.NforcePoints,
        preparedOPT5Homog.NForceField, length(sourceTimesOPTHomog))
    for sourceNode in sourceNodesOPTHomog
        sourceOPT5Homog[sourceNode.linear, 2, :] .+=
            sourceNode.weight .* sourceScaleOPTHomog .* waveletOPTHomog
    end
    propagationOPT5Homog = propagateLinearSystem(
        preparedOPT5Homog, optStepsHomog, dtOPTHomog;
        sourceFull=sourceOPT5Homog, output_stride=optOutputStrideHomog,
        blowup_limit=1e8, solver_name="OPT5",
    )
    @assert !propagationOPT5Homog.stopped_early
    uxOPT5Homog = propagationOPT5Homog.history[:, :, 1, :]
    uzOPT5Homog = propagationOPT5Homog.history[:, :, 2, :]
    opt5TimesHomog = propagationOPT5Homog.times
else
    propagationOPT5Homog = uxOPT5Homog = uzOPT5Homog = opt5TimesHomog = nothing
end

# Run the less stable delta-test conventional recipe last, so an eventual
# late stop cannot discard the expensive OPT5 result from this session.
sourceConvFDHomog = zeros(Float64, preparedConvFDHomog.NforcePoints,
    preparedConvFDHomog.NForceField, length(sourceTimesOPTHomog))
for sourceNode in sourceNodesOPTHomog
    sourceConvFDHomog[sourceNode.linear, 2, :] .+=
        sourceNode.weight .* sourceScaleOPTHomog .* waveletOPTHomog
end
# orderB=-1 is an interior conventional stencil, not a stable closure for
# the abruptly truncated outer rowsHomog. Stop before the earliest source tail
# can excite those rowsHomog unless the user explicitly requests the diagnostic.
convFDBoundaryGuardHomog = 0.5 / sourceFrequency
convFDBoundarySafeTimeHomog = max(0.0, earliestPEdgeArrivalHomog - convFDBoundaryGuardHomog)
convFDDurationHomog = runConvFDBeyondBoundarySafeTime ? duration :
    min(duration, convFDBoundarySafeTimeHomog)
convFDStepsHomog = ceil(Int, convFDDurationHomog / dtOPTHomog)
propagationConvFDHomog = propagateLinearSystem(
    preparedConvFDHomog, convFDStepsHomog, dtOPTHomog;
    sourceFull=sourceConvFDHomog, output_stride=optOutputStrideHomog,
    blowup_limit=1e8,
    solver_name="convFD3 (orderB=-1)",
)
uxConvFDHomog = propagationConvFDHomog.history[:, :, 1, :]
uzConvFDHomog = propagationConvFDHomog.history[:, :, 2, :]
convFDTimesHomog = propagationConvFDHomog.times
propagationConvFDHomog.stopped_early && @warn(
    "convFD3 stopped early; its partial history remains available",
    final_time=last(convFDTimesHomog), requested_duration=duration)
@show convFDBoundarySafeTimeHomog convFDDurationHomog propagationConvFDHomog.stopped_early
@show dtOPTHomog size(uzOPTHomog) maximum(abs, uxOPTHomog) maximum(abs, uzOPTHomog)
@show size(uzConvFDHomog) maximum(abs, uxConvFDHomog) maximum(abs, uzConvFDHomog)
runOPT5 && @show size(uzOPT5Homog) maximum(abs, uxOPT5Homog) maximum(abs, uzOPT5Homog)



In [ ]:
fdConfigHomog = ElasticThreePointConfig2D(
    pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, cfl=0.38,
)
fdHomog = prepare_elastic_wave_2d(
    modelHomog, (dx, dx);
    material_mask=material,
    boundary_conditions=bcFD,
    config=fdConfigHomog,
)
fdCoordinatesHomog = elastic_wave_coordinates(x, z, fdHomog)
fdPaddingHomog = fdHomog.padding
sourcePhysicalHomog = CartesianIndex(
    argmin(abs.(x .- sourcePosition.x)),
    argmin(abs.(z .- sourcePosition.z)),
)
sourceFDHomog = sourcePhysicalHomog + CartesianIndex(Tuple(fdPaddingHomog[1, :]))
sourceSpecsFDHomog = spatial_source_specs(fdCoordinatesHomog.x, fdCoordinatesHomog.z)
sourceNodesFDHomog = [(index=CartesianIndex(
    argmin(abs.(fdCoordinatesHomog.x .- s.x)),
    argmin(abs.(fdCoordinatesHomog.z .- s.z))), weight=s.weight)
    for s in sourceSpecsFDHomog]
fdStepsHomog = ceil(Int, duration / fdHomog.dt)
fdOutputStrideHomog = max(1, round(Int, outputSampling / fdHomog.dt))
fdFramesXHomog = Matrix{Float32}[copy(fdHomog.ux)]
fdFramesZHomog = Matrix{Float32}[copy(fdHomog.uz)]
fdTimesHomog = Float64[0.0]
for step in 1:fdStepsHomog
    step_elastic_wave_2d!(fdHomog)
    for sourceNode in sourceNodesFDHomog
        elasticWave2D.add_ricker_source!(
            fdHomog, sourceNode.index;
            f0=sourceFrequency, t0=sourceDelay,
            amplitude=sourceForce * sourceNode.weight,
            component=:z, source_kind=:force,
        )
    end
    if step % fdOutputStrideHomog == 0 || step == fdStepsHomog
        push!(fdFramesXHomog, copy(fdHomog.ux))
        push!(fdFramesZHomog, copy(fdHomog.uz)); push!(fdTimesHomog, fdHomog.time)
    end
end
uxFDHomog = cat(fdFramesXHomog...; dims=3)
uzFDHomog = cat(fdFramesZHomog...; dims=3)
@show fdHomog.dt size(uzFDHomog) maximum(abs, uxFDHomog) maximum(abs, uzFDHomog)



In [ ]:
waveformQuantity = @isdefined(waveformQuantity) ? waveformQuantity : :velocity
function displacement_tracesHomog(history, times, xaxis, zindex, receivers)
    tracesHomog = Matrix{Float64}(undef, length(times), length(receivers))
    for (j, receiverHomog) in enumerate(receivers)
        ixHomog = argmin(abs.(xaxis .- receiverHomog))
        tracesHomog[:, j] .= Float64.(history[ixHomog, zindex, :])
    end
    return (time=Float64.(times), values=tracesHomog)
end

function velocity_tracesHomog(history, times, xaxis, zindex, receivers)
    tracesHomog = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiverHomog) in enumerate(receivers)
        ixHomog = argmin(abs.(xaxis .- receiverHomog))
        tracesHomog[:, j] .= diff(Float64.(history[ixHomog, zindex, :])) ./ diff(times)
    end
    return (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=tracesHomog)
end
function velocity_traces_at_pointsHomog(history, times, xaxis, zaxis, receivers;
    sampling=:nearest)
    sampling === :nearest || error("only :nearest is enabled in this cheap benchmark path")
    tracesHomog = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiverHomog) in enumerate(receivers)
        ixHomog = argmin(abs.(xaxis .- receiverHomog.x))
        izHomog = argmin(abs.(zaxis .- receiverHomog.z))
        tracesHomog[:, j] .= diff(Float64.(history[ixHomog, izHomog, :])) ./ diff(times)
    end
    (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=tracesHomog,
        receivers=collect(receivers), sampling)
end
fdReceiverZIndexHomog = argmin(abs.(fdCoordinatesHomog.z .- receiverZ))
optReceiverZIndexHomog = argmin(abs.(zOPTHomog .- receiverZ))
fdDisplacementXHomog = displacement_tracesHomog(uxFDHomog, fdTimesHomog, fdCoordinatesHomog.x,
    fdReceiverZIndexHomog, receiverX)
fdDisplacementZHomog = displacement_tracesHomog(uzFDHomog, fdTimesHomog, fdCoordinatesHomog.x,
    fdReceiverZIndexHomog, receiverX)
optDisplacementXHomog = displacement_tracesHomog(uxOPTHomog, optTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
optDisplacementZHomog = displacement_tracesHomog(uzOPTHomog, optTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
convFDDisplacementXHomog = displacement_tracesHomog(uxConvFDHomog, convFDTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
convFDDisplacementZHomog = displacement_tracesHomog(uzConvFDHomog, convFDTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
optNoGammaDisplacementXHomog = runOPTWithoutGamma ? displacement_tracesHomog(
    uxOPTWithoutGammaHomog, optTimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
optNoGammaDisplacementZHomog = runOPTWithoutGamma ? displacement_tracesHomog(
    uzOPTWithoutGammaHomog, optTimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
opt4DisplacementXHomog = runOPT4 ? displacement_tracesHomog(
    uxOPT4Homog, opt4TimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
opt4DisplacementZHomog = runOPT4 ? displacement_tracesHomog(
    uzOPT4Homog, opt4TimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
opt5DisplacementXHomog = runOPT5 ? displacement_tracesHomog(
    uxOPT5Homog, opt5TimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
opt5DisplacementZHomog = runOPT5 ? displacement_tracesHomog(
    uzOPT5Homog, opt5TimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing

fdVelocityXHomog = velocity_tracesHomog(uxFDHomog, fdTimesHomog, fdCoordinatesHomog.x,
    fdReceiverZIndexHomog, receiverX)
fdVelocityZHomog = velocity_tracesHomog(uzFDHomog, fdTimesHomog, fdCoordinatesHomog.x,
    fdReceiverZIndexHomog, receiverX)
optVelocityXHomog = velocity_tracesHomog(uxOPTHomog, optTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
optVelocityZHomog = velocity_tracesHomog(uzOPTHomog, optTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
convFDVelocityXHomog = velocity_tracesHomog(uxConvFDHomog, convFDTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
convFDVelocityZHomog = velocity_tracesHomog(uzConvFDHomog, convFDTimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX)
optNoGammaVelocityXHomog = runOPTWithoutGamma ? velocity_tracesHomog(
    uxOPTWithoutGammaHomog, optTimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
optNoGammaVelocityZHomog = runOPTWithoutGamma ? velocity_tracesHomog(
    uzOPTWithoutGammaHomog, optTimesHomog, xOPTHomog, optReceiverZIndexHomog, receiverX) : nothing
opt4VelocityXHomog = runOPT4 ? velocity_tracesHomog(uxOPT4Homog, opt4TimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX) : nothing
opt4VelocityZHomog = runOPT4 ? velocity_tracesHomog(uzOPT4Homog, opt4TimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX) : nothing
opt5VelocityXHomog = runOPT5 ? velocity_tracesHomog(uxOPT5Homog, opt5TimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX) : nothing
opt5VelocityZHomog = runOPT5 ? velocity_tracesHomog(uzOPT5Homog, opt5TimesHomog, xOPTHomog,
    optReceiverZIndexHomog, receiverX) : nothing
if waveformQuantity === :velocity
    fdWaveformXHomog, fdWaveformZHomog = fdVelocityXHomog, fdVelocityZHomog
    optWaveformXHomog, optWaveformZHomog = optVelocityXHomog, optVelocityZHomog
    convFDWaveformXHomog, convFDWaveformZHomog = convFDVelocityXHomog, convFDVelocityZHomog
    optNoGammaWaveformXHomog, optNoGammaWaveformZHomog =
        optNoGammaVelocityXHomog, optNoGammaVelocityZHomog
    opt4WaveformXHomog, opt4WaveformZHomog = opt4VelocityXHomog, opt4VelocityZHomog
    opt5WaveformXHomog, opt5WaveformZHomog = opt5VelocityXHomog, opt5VelocityZHomog
    waveformUnitLabelHomog, waveformComponentPrefixHomog = "m/s", "v"
else
    fdWaveformXHomog, fdWaveformZHomog = fdDisplacementXHomog, fdDisplacementZHomog
    optWaveformXHomog, optWaveformZHomog = optDisplacementXHomog, optDisplacementZHomog
    convFDWaveformXHomog, convFDWaveformZHomog = convFDDisplacementXHomog, convFDDisplacementZHomog
    optNoGammaWaveformXHomog, optNoGammaWaveformZHomog =
        optNoGammaDisplacementXHomog, optNoGammaDisplacementZHomog
    opt4WaveformXHomog, opt4WaveformZHomog = opt4DisplacementXHomog, opt4DisplacementZHomog
    opt5WaveformXHomog, opt5WaveformZHomog = opt5DisplacementXHomog, opt5DisplacementZHomog
    waveformUnitLabelHomog, waveformComponentPrefixHomog = "m", "u"
end
fdVelocityHomog, optVelocityHomog = fdVelocityZHomog, optVelocityZHomog # physical velocity aliases
receiverHomog = cld(length(receiverX), 2)
fdTraceHomog = (time=fdWaveformZHomog.time, values=fdWaveformZHomog.values[:, receiverHomog])
optTraceHomog = (time=optWaveformZHomog.time, values=optWaveformZHomog.values[:, receiverHomog])
convFDTraceHomog = (time=convFDWaveformZHomog.time,
    values=convFDWaveformZHomog.values[:, receiverHomog])
optNoGammaTraceHomog = runOPTWithoutGamma ? (time=optNoGammaWaveformZHomog.time,
    values=optNoGammaWaveformZHomog.values[:, receiverHomog]) : nothing
opt4TraceHomog = runOPT4 ? (time=opt4WaveformZHomog.time,
    values=opt4WaveformZHomog.values[:, receiverHomog]) : nothing
opt5TraceHomog = runOPT5 ? (time=opt5WaveformZHomog.time,
    values=opt5WaveformZHomog.values[:, receiverHomog]) : nothing
preSPECFEMTracesHomog = (explicitFD3=fdTraceHomog,
    convFD3=convFDTraceHomog, OPT3=optTraceHomog)
runOPTWithoutGamma && (preSPECFEMTracesHomog = merge(preSPECFEMTracesHomog,
    (OPT3_no_Gamma=optNoGammaTraceHomog,)))
runOPT4 && (preSPECFEMTracesHomog = merge(preSPECFEMTracesHomog, (OPT4=opt4TraceHomog,)))
runOPT5 && (preSPECFEMTracesHomog = merge(preSPECFEMTracesHomog, (OPT5=opt5TraceHomog,)))
preSPECFEMHomog = plot_solver_benchmark(preSPECFEMTracesHomog;
    title="Heterogeneous interior modelHomog — normalized $(waveformComponentPrefixHomog)_z")
display(preSPECFEMHomog.figure)
nothing



In [ ]:
dtSPECFEMHomog = dtOPTHomog
caseDirectoryHomog = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "homogeneous_kirishima_no_free_surface")
solidZHomog = applyFreeSurface ? findall(z .<= 0.0) : collect(eachindex(z))
zSPECFEMHomog = z[solidZHomog]
vpHomog = vpFieldHomog[:, solidZHomog]
vsHomog = vsFieldHomog[:, solidZHomog]
rhoHomog = rhoFieldHomog[:, solidZHomog]
# Refining FD/OPT must not silently refine the SPECFEM spectral mesh.
specfemNxElementsHomog = round(Int, (last(x)-first(x)) / (4referenceDx))
specfemNzElementsHomog = round(Int, (last(zSPECFEMHomog)-first(zSPECFEMHomog)) / (4referenceDx))
specfemCaseHomog = prepare_specfem2d_case(
    caseDirectoryHomog, x, zSPECFEMHomog, vpHomog, vsHomog, rhoHomog, fill(last(zSPECFEMHomog), nx);
    source=sourcePosition, sources=sourceSpecsReference, receivers=receiverX,
    duration=duration, dt=dtSPECFEMHomog, f0=sourceFrequency,
    source_factor=sourceForce, source_angle=0.0,
    source_time_function=rickerSource,
    nx_elements=specfemNxElementsHomog, nz_elements=specfemNzElementsHomog,
    receiver_z=receiverZ, free_surface=applyFreeSurface,
    snapshot_interval_steps=max(1, round(Int, outputSampling / dtSPECFEMHomog)),
    snapshot_image_type=5, # vertical velocity, matching the seismograms
    output_wavefield_dumps=true,
    wavefield_dump_type=1, # displacementHomog vector (uₓ, u_z)
    binary_wavefield_dumps=true,
)
if runSPECFEM2D
    specfemRunHomog = run_specfem2d_case(specfemCaseHomog.case_directory)
else
    specfemRunHomog = (output=specfemCaseHomog.output,)
end
verticalFilesHomog = find_specfem2d_traces(specfemRunHomog.output; component=:z)
@assert length(verticalFilesHomog) == length(receiverX)
specfemTracesZHomog = [read_specfem2d_trace(file;
    time_shift=specfemCaseHomog.time_axis_shift) for file in verticalFilesHomog]
horizontalFilesHomog = find_specfem2d_traces(specfemRunHomog.output; component=:x)
@assert length(horizontalFilesHomog) == length(receiverX)
specfemTracesXHomog = [read_specfem2d_trace(file;
    time_shift=specfemCaseHomog.time_axis_shift) for file in horizontalFilesHomog]
function integrate_velocity_traceHomog(trace)
    displacementHomog = zeros(Float64, length(trace.values))
    displacementHomog[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .*
        diff(trace.time))
    (time=Float64.(trace.time), values=displacementHomog)
end
specfemWaveformsXHomog = waveformQuantity === :velocity ? specfemTracesXHomog :
    integrate_velocity_traceHomog.(specfemTracesXHomog)
specfemWaveformsZHomog = waveformQuantity === :velocity ? specfemTracesZHomog :
    integrate_velocity_traceHomog.(specfemTracesZHomog)
specfemTracesHomog = specfemWaveformsZHomog # selected-quantity compatibility alias
specfemTraceHomog = specfemWaveformsZHomog[receiverHomog]
specfemWavefieldHomog = read_specfem2d_wavefield_dumps(
    specfemRunHomog.output; dt=dtSPECFEMHomog,
    time_axis_shift=specfemCaseHomog.time_axis_shift,
)
specfemSnapshotVideoHomog = make_specfem2d_snapshot_video(
    specfemRunHomog.output;
    output_path=joinpath(specfemRunHomog.output, "SPECFEM2D_vertical_velocity.mp4"),
    framerate=20,
)
solverLogTextHomog = read(joinpath(specfemCaseHomog.case_directory, "solver.log"), String)
specfemCFLMatchHomog = match(r"Max CFL stability condition[^=]*=\s*([0-9.Ee+-]+)",
    solverLogTextHomog)
specfemCFLHomog = isnothing(specfemCFLMatchHomog) ? missing :
    parse(Float64, specfemCFLMatchHomog.captures[1])
cflSummaryHomog = (
    FD3_axis=vpMaximumHomog * Float64(fdHomog.dt) / dx,
    FD3_2D=sqrt(2) * vpMaximumHomog * Float64(fdHomog.dt) / dx,
    OPT3_axis=vpMaximumHomog * dtOPTHomog / dxOPTHomog,
    OPT3_2D=sqrt(2) * vpMaximumHomog * dtOPTHomog / dxOPTHomog,
    SPECFEM2D_reported=specfemCFLHomog,
)
# SPECFEM uses 5 GLL points per element in this build. Its default number
# of elements makes the mean GLL-node interval comparable to dx.
specfemElementSizeHomog = (
    x=(last(x)-first(x)) / specfemCaseHomog.nx_elements,
    z=(last(zSPECFEMHomog)-first(zSPECFEMHomog)) / specfemCaseHomog.nz_elements,
)
discretizationSummaryHomog = (
    OPT3=(dx=dxOPTHomog, dt=dtOPTHomog),
    SPECFEM2D=(element_size=specfemElementSizeHomog,
        mean_GLL_interval=(x=specfemElementSizeHomog.x/4,
                           z=specfemElementSizeHomog.z/4), dt=dtSPECFEMHomog),
)
@show cflSummaryHomog discretizationSummaryHomog specfemCaseHomog.time_axis_shift
@show specfemCaseHomog.case_directory dtSPECFEMHomog specfemSnapshotVideoHomog.path
if endswith(lowercase(specfemSnapshotVideoHomog.path), ".gif")
    gifDataHomog = base64encode(read(specfemSnapshotVideoHomog.path))
    display(MIME("text/html"),
        "<img src='data:image/gif;base64,$(gifDataHomog)' style='max-width:100%'>")
else
    # VS Code/IJulia renders the MP4 with native playback controls.
    videoURLHomog = "file://" * specfemSnapshotVideoHomog.path
    display(MIME("text/html"),
        "<video controls loop src='$(videoURLHomog)' style='max-width:100%'></video>")
end
specfemSnapshotVideoHomog.path



In [ ]:
function compact_waveform_metricsHomog(referenceHomog, candidate)
    fullHomog = waveform_metrics(referenceHomog, candidate; samples=1001)
    (correlation=fullHomog.correlation, relative_error=fullHomog.relative_error,
     relative_rmse=fullHomog.relative_rmse,
     optimal_amplitude=fullHomog.optimal_amplitude,
     bias=fullHomog.bias, error_variance=fullHomog.error_variance)
end

function solver_traces_for_stationHomog(station)
    tracesHomog = (
        FD3=(time=fdWaveformZHomog.time, values=fdWaveformZHomog.values[:, station]),
        convFD3=(time=convFDWaveformZHomog.time,
            values=convFDWaveformZHomog.values[:, station]),
        OPT3=(time=optWaveformZHomog.time, values=optWaveformZHomog.values[:, station]),
    )
    runOPT4 && (tracesHomog = merge(tracesHomog, (OPT4=(time=opt4WaveformZHomog.time,
        values=opt4WaveformZHomog.values[:, station]),)))
    runOPT5 && (tracesHomog = merge(tracesHomog, (OPT5=(time=opt5WaveformZHomog.time,
        values=opt5WaveformZHomog.values[:, station]),)))
    tracesHomog
end

stationErrorMetricsHomog = map(eachindex(receiverX)) do station
    referenceHomog = specfemWaveformsZHomog[station]
    candidatesHomog = solver_traces_for_stationHomog(station)
    metricsHomog = map(trace -> compact_waveform_metricsHomog(referenceHomog, trace), candidatesHomog)
    (station=station, x_km=receiverX[station]/1e3, metrics=metricsHomog)
end
foreach(display, stationErrorMetricsHomog)

methodNamesHomog = keys(first(stationErrorMetricsHomog).metrics)
globalErrorMetricsHomog = NamedTuple{methodNamesHomog}(map(methodNamesHomog) do method
    rowsHomog = [getproperty(item.metrics, method) for item in stationErrorMetricsHomog]
    (mean_correlation=mean(getproperty.(rowsHomog, :correlation)),
     rms_relative_error=sqrt(mean(abs2, getproperty.(rowsHomog, :relative_error))),
     maximum_relative_error=maximum(getproperty.(rowsHomog, :relative_error)),
     mean_raw_relative_rmse=mean(getproperty.(rowsHomog, :relative_rmse)))
end)
display(globalErrorMetricsHomog)

errorFigureHomog = Figure(size=(1050, 430))
correlationAxisHomog = Axis(errorFigureHomog[1, 1]; xlabel="receiverHomog x (km)",
    ylabel="correlation with SPECFEM2D")
relativeAxisHomog = Axis(errorFigureHomog[1, 2]; xlabel="receiverHomog x (km)",
    ylabel="shape-relative error")
for method in methodNamesHomog
    rowsHomog = [getproperty(item.metrics, method) for item in stationErrorMetricsHomog]
    xsHomog = getproperty.(stationErrorMetricsHomog, :x_km)
    scatterlines!(correlationAxisHomog, xsHomog, getproperty.(rowsHomog, :correlation);
        marker=:circle, label=string(method))
    scatterlines!(relativeAxisHomog, xsHomog, getproperty.(rowsHomog, :relative_error);
        marker=:circle, label=string(method))
end
axislegend(correlationAxisHomog); axislegend(relativeAxisHomog)
errorFigureHomog



### Heterogeneous sub-case


In [ ]:
# Heterogeneous sub-case: the real Kirishima NIED velocity/density field
# over the Step 2 box, reused verbatim (km/s, g/cm^3 -> m/s, kg/m^3).
vpFieldHeter = seismicModel2DStep2.Vpv .* 1e3
vsFieldHeter = seismicModel2DStep2.Vsv .* 1e3
rhoFieldHeter = seismicModel2DStep2.ρ .* 1e3
modelHeter = (ρ=rhoFieldHeter ./ 1e3, Vpv=vpFieldHeter ./ 1e3, Vsv=vsFieldHeter ./ 1e3)
vpMaximumHeter, vsMinimumHeter = maximum(vpFieldHeter), minimum(vsFieldHeter)
pointsPerSWavelengthHeter = (
    FD3=vsMinimumHeter / (sourceFrequency * dx),
    OPT3=vsMinimumHeter / (sourceFrequency * (dx / optSpatialRefinement)),
    OPT5=vsMinimumHeter / (sourceFrequency * (dx / optSpatialRefinement)),
)
earliestPEdgeArrivalHeter = significantSourceStart +
    distanceToNearestPhysicalEdge / vpMaximumHeter
@assert duration < earliestPEdgeArrivalHeter
@assert minimum(values(pointsPerSWavelengthHeter)) >=
    minimumAcceptedPointsPerSWavelength
@show vpMaximumHeter vsMinimumHeter pointsPerSWavelengthHeter earliestPEdgeArrivalHeter


In [ ]:
dxOPTHeter = dx / optSpatialRefinement
xOPTPhysicalHeter = collect(first(x):dxOPTHeter:last(x))
zOPTPhysicalHeter = collect(first(z):dxOPTHeter:last(z))
solidOPTHeter = applyFreeSurface ?
    repeat(reshape(zOPTPhysicalHeter .<= 0.0, 1, :),
        length(xOPTPhysicalHeter), 1) :
    trues(length(xOPTPhysicalHeter), length(zOPTPhysicalHeter))
dtOPTHeter = 0.20 * dxOPTHeter / (sqrt(2) * vpMaximumHeter * temporalRefinement)
@assert optSpatialRefinement == 1 "OPT refinement != 1 needs field resampling, not implemented for the Kirishima case yet"
vpFieldOPTHeter, vsFieldOPTHeter, rhoFieldOPTHeter = vpFieldHeter, vsFieldHeter, rhoFieldHeter
# Stable referenceHeter used by the previously working OPT benchmark. This is
# an exact change of coordinates x'=x/Δx, z'=z/Δz, t'=t/Δt, not a
# change of the physical modelHeter. The direct physical-recipe path remains
# available for conditioning experiments, but is not the benchmark default.
optFormulationHeter = :scaled_coordinates # or :physical_recipe_experimental
if optFormulationHeter === :scaled_coordinates
    recipeSpacingHeter = (1.0, 1.0, 1.0)
    rhoOPTHeter = rhoFieldOPTHeter ./ rho0
    muOPTHeter = (rhoFieldOPTHeter .* vsFieldOPTHeter.^2 ./ rho0) .* (dtOPTHeter/dxOPTHeter)^2
    lambdaOPTHeter = (rhoFieldOPTHeter .* (vpFieldOPTHeter.^2 .- 2vsFieldOPTHeter.^2) ./ rho0) .*
        (dtOPTHeter/dxOPTHeter)^2
elseif optFormulationHeter === :physical_recipe_experimental
    recipeSpacingHeter = (dxOPTHeter, dxOPTHeter, dtOPTHeter)
    rhoOPTHeter = rhoFieldOPTHeter
    muOPTHeter = rhoFieldOPTHeter .* vsFieldOPTHeter.^2
    lambdaOPTHeter = rhoFieldOPTHeter .* (vpFieldOPTHeter.^2 .- 2vsFieldOPTHeter.^2)
else
    error("unknown optFormulation=$optFormulationHeter")
end
muOPTHeter[.!solidOPTHeter] .= 0.0
lambdaOPTHeter[.!solidOPTHeter] .= 0.0
optParametersHeter = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
    "Δ" => recipeSpacingHeter,
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    # Preserve all total-degree <= 2 Taylor conditions exactly; use
    # supplementaryOrder only to optimise the remaining null space.
    "taylorInverseMode" => :hierarchical_constrained,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    # The expensive semi-symbolic coefficientHeter construction runs on the
    # GPU selected in the bootstrap cell (Metal here).
    "recipe_backend" => backend,
)
# Cache complete semi-symbolic recipes, not the very large prepared
# operators. The runtime Metal/CPU object is deliberately excluded from
# the cache key and is injected only when a recipe must be produced.
recipeCacheDirectoryHeter = "semiSymbolic"
recipeCacheVersionHeter = 3 # hierarchical Taylor inverse; reject legacy recipes
function cachedOPTRecipeHeter(parameters, prefix; existing=nothing)
    cacheParametersHeter = Dict{String,Any}(
        key => value for (key, value) in parameters
        if key != "recipe_backend"
    )
    cacheParametersHeter["recipe_cache_version"] = recipeCacheVersionHeter
    function produceRecipeHeter(config)
        runtimeParametersHeter = Dict{String,Any}(config)
        pop!(runtimeParametersHeter, "hash_id", nothing)
        pop!(runtimeParametersHeter, "recipe_cache_version", nothing)
        runtimeParametersHeter["recipe_backend"] = backend
        isnothing(existing) ?
            makeOPTsemiSymbolic(runtimeParametersHeter) : existing
    end
    myProduceOrLoad(produceRecipeHeter, cacheParametersHeter,
        recipeCacheDirectoryHeter, prefix)
end
existingOPT3RecipeHeter = @isdefined(optRecipeHeter) ? optRecipeHeter : nothing
optRecipeHeter = cachedOPTRecipeHeter(optParametersHeter, "elastic2D_OPT3";
    existing=existingOPT3RecipeHeter)
# P-C predictor: same three-point support and physical modelHeter, but only
# the supplementary Taylor optimisation is removed. The fullHeter OPT3
# operator remains the residual operator used by every correction.
optPredictorParametersHeter = copy(optParametersHeter)
optPredictorParametersHeter["supplementaryOrder"] = 0
optPredictorRecipeHeter = optTimeMarchingScheme === :predictor_corrector &&
    optPredictorMode === :prepared ?
    cachedOPTRecipeHeter(optPredictorParametersHeter, "elastic2D_OPT3_predictor_s0") : nothing

# Four/five-point spatial recipes; time remains a three-point stencil.
opt4ParametersHeter = copy(optParametersHeter)
opt4ParametersHeter["pointsInSpace"] = 4
opt4ParametersHeter["fieldItpl"] = merge(optParametersHeter["fieldItpl"],
    (offsetSpace=1.5,))
opt4ParametersHeter["materItpl"] = merge(optParametersHeter["materItpl"],
    (offsetSpace=1.5,))
opt4RecipeHeter = buildHigherOrderRecipes ? cachedOPTRecipeHeter(
    opt4ParametersHeter, "elastic2D_OPT4"; existing=nothing) : nothing

# Five-point OPT recipe: the time stencil remains three points so that
# only the spatial approximation changes relative to OPT3.
opt5ParametersHeter = copy(optParametersHeter)
# Validated OPT5 family used by seismo1Dbenchmark: five trial points
# retain the linear B-spline test order. orderBspace=3 is not OPT5 and
# generated a nearly singular first-step solve in this elastic problem.
opt5ParametersHeter["orderBspace"] = 1
opt5ParametersHeter["pointsInSpace"] = 5
opt5ParametersHeter["fieldItpl"] = merge(optParametersHeter["fieldItpl"],
    (offsetSpace=2.0,))
opt5ParametersHeter["materItpl"] = merge(optParametersHeter["materItpl"],
    (offsetSpace=2.0,))
# Keep OPT5 available, but do not even load/build it in the quick run.
# When enabled, never migrate the former invalid orderBspace=3 object.
opt5RecipeHeter = buildHigherOrderRecipes ? cachedOPTRecipeHeter(
    opt5ParametersHeter, "elastic2D_OPT5"; existing=nothing) : nothing

# Collocation/delta-test recipe corresponding to the conventional FD3
# definition used in seismo1Dbenchmark.ipynb. Keep it beside OPT3: it is
# a coefficientHeter referenceHeter, not an alias for the explicit flux solver.
conventionalFDParametersHeter = copy(optParametersHeter)
conventionalFDParametersHeter["orderBspace"] = -1
conventionalFDParametersHeter["orderBtime"] = -1
conventionalFDParametersHeter["supplementaryOrder"] = 0
existingConvFD3RecipeHeter = @isdefined(conventionalFDRecipeHeter) ?
    conventionalFDRecipeHeter : nothing
conventionalFDRecipeHeter = cachedOPTRecipeHeter(
    conventionalFDParametersHeter, "elastic2D_convFD3";
    existing=existingConvFD3RecipeHeter)
recipeComparisonHeter = (
    conventionalFD3=(orderBspace=-1, orderBtime=-1,
        supplementaryOrder=0, pointsInSpace=3, pointsInTime=3),
    OPT3=(orderBspace=optParametersHeter["orderBspace"],
        orderBtime=optParametersHeter["orderBtime"],
        supplementaryOrder=optParametersHeter["supplementaryOrder"],
        pointsInSpace=optParametersHeter["pointsInSpace"],
        pointsInTime=optParametersHeter["pointsInTime"]),
    OPT4=(orderBspace=opt4ParametersHeter["orderBspace"],
        orderBtime=opt4ParametersHeter["orderBtime"],
        supplementaryOrder=opt4ParametersHeter["supplementaryOrder"],
        pointsInSpace=opt4ParametersHeter["pointsInSpace"],
        pointsInTime=opt4ParametersHeter["pointsInTime"]),
    OPT5=(orderBspace=opt5ParametersHeter["orderBspace"],
        orderBtime=opt5ParametersHeter["orderBtime"],
        supplementaryOrder=opt5ParametersHeter["supplementaryOrder"],
        pointsInSpace=opt5ParametersHeter["pointsInSpace"],
        pointsInTime=opt5ParametersHeter["pointsInTime"]),
)
# Public handles for inspecting the complete symbolic recipes in IJulia,
# e.g. `optRecipesHeter.OPT5["recette"]`.
optRecipesHeter = (convFD3=conventionalFDRecipeHeter, OPT3=optRecipeHeter,
    OPT4=opt4RecipeHeter, OPT5=opt5RecipeHeter)
@show recipeComparisonHeter
if runOPT4 || runOPT5
    coefficientGatePathHeter = joinpath(flexopt_root, "data",
        "elastic_lhs_coefficient_gate.jld2")
    isfile(coefficientGatePathHeter) || error(
        "Run ElasticLHSOperatorAudit.ipynb before OPT4/OPT5 propagation")
    coefficientGateHeter = load(coefficientGatePathHeter)
    @assert all(result.passed for result in
        values(coefficientGateHeter["coefficient_health"]))
end



In [ ]:
optCerjanHeter = CerjanBoundarySpec(
    (cerjan.lower[1] * optSpatialRefinement,
        cerjan.lower[2] * optSpatialRefinement),
    (cerjan.upper[1] * optSpatialRefinement, 0);
    damping=cerjan.damping / optSpatialRefinement^2,
)
activeOptCerjanHeter = applyCerjan ? optCerjanHeter : nothing
bcOPTHeter = applyFreeSurface ?
    boundary_geometry(solidOPTHeter, (dxOPTHeter, dxOPTHeter);
        free_surface_mode=:pinned_void, cerjan=activeOptCerjanHeter) :
    BoundaryConditionSet(free_surface=nothing, cerjan=activeOptCerjanHeter,
        material_mask=BitArray(solidOPTHeter), free_surface_mode=:pinned_void)
modelsOPTHeter = [rhoOPTHeter, lambdaOPTHeter, muOPTHeter]
pointsOPTHeter = getModelPoints(modelsOPTHeter[1], 3,
    optRecipeHeter["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
familyOPTHeter = (models=modelsOPTHeter, modelPoints=pointsOPTHeter,
    Δ=recipeSpacingHeter, modelName="homogeneous_OPT3_$(optFormulationHeter)")
operatorTimingsHeter = Dict{Symbol,Float64}()
timedOPT3AssemblyHeter = @timed numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => optRecipeHeter, "modelFam" => familyOPTHeter,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPTHeter,
    "representation" => "matrixfree",
))["numOperators"]
numericalVolumeHeter = timedOPT3AssemblyHeter.value
operatorTimingsHeter[:OPT3_assembly] = timedOPT3AssemblyHeter.time
timedOPT3PreparationHeter = @timed prepareLinearSystem(numericalVolumeHeter;
    free_surface_spacing=recipeSpacingHeter[1:2])
preparedVolumeHeter = timedOPT3PreparationHeter.value
operatorTimingsHeter[:OPT3_preparation] = timedOPT3PreparationHeter.time
preparedOPT4VolumeHeter = if runOPT4
    pointsOPT4Heter = getModelPoints(modelsOPTHeter[1], 4,
        opt4RecipeHeter["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT4Heter = (models=modelsOPTHeter, modelPoints=pointsOPT4Heter,
        Δ=recipeSpacingHeter, modelName="homogeneous_OPT4_$(optFormulationHeter)")
    numericalOPT4Heter = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt4RecipeHeter, "modelFam" => familyOPT4Heter,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPTHeter, "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numericalOPT4Heter; free_surface_spacing=recipeSpacingHeter[1:2])
else
    nothing
end
preparedOPT5VolumeHeter = if runOPT5
    pointsOPT5Heter = getModelPoints(modelsOPTHeter[1], 5,
        opt5RecipeHeter["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT5Heter = (models=modelsOPTHeter, modelPoints=pointsOPT5Heter,
        Δ=recipeSpacingHeter, modelName="homogeneous_OPT5_$(optFormulationHeter)")
    numericalOPT5Heter = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt5RecipeHeter, "modelFam" => familyOPT5Heter,
        "absorbingBoundaries" => nothing,
        "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPTHeter,
        "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numericalOPT5Heter;
        free_surface_spacing=recipeSpacingHeter[1:2])
else
    nothing
end
timedConvFDAssemblyHeter = @timed numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => conventionalFDRecipeHeter, "modelFam" => familyOPTHeter,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPTHeter,
    "representation" => "matrixfree",
))["numOperators"]
numericalConvFDHeter = timedConvFDAssemblyHeter.value
operatorTimingsHeter[:convFD3_assembly] = timedConvFDAssemblyHeter.time
timedConvFDPreparationHeter = @timed prepareLinearSystem(numericalConvFDHeter;
    free_surface_spacing=recipeSpacingHeter[1:2])
preparedConvFDVolumeHeter = timedConvFDPreparationHeter.value
operatorTimingsHeter[:convFD3_preparation] = timedConvFDPreparationHeter.time

# Optional traction recipe σn=0, assembled additively on surface rowsHeter.
if applyFreeSurface
boundaryParametersHeter = copy(optParametersHeter)
boundaryParametersHeter["famousEquationType"] = "elasticTractionFree2D"
boundaryRecipeHeter = cachedOPTRecipeHeter(
    boundaryParametersHeter, "elasticTractionFree2D_OPT3")
normalXHeter = zeros(size(solidOPTHeter)); normalZ = zeros(size(solidOPTHeter))
for (point, normal) in zip(bcOPTHeter.free_surface.points, bcOPTHeter.free_surface.normals)
    normalXHeter[point], normalZ[point] = normal
end
boundaryModelsHeter = [lambdaOPTHeter, muOPTHeter, normalXHeter, normalZ]
boundaryPointsHeter = getModelPoints(boundaryModelsHeter[1], 3,
    boundaryRecipeHeter["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
boundaryFamilyHeter = (models=boundaryModelsHeter, modelPoints=boundaryPointsHeter,
    Δ=recipeSpacingHeter, modelName="homogeneous_free_surface")
numericalBoundaryHeter = numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => boundaryRecipeHeter, "modelFam" => boundaryFamilyHeter,
    "absorbingBoundaries" => (isnothing(bcOPTHeter.cerjan) ? nothing : cerjan_padding(bcOPTHeter.cerjan)),
    "maskedRegionInSpace" => bcOPTHeter.free_surface.points,
    "representation" => "matrixfree",
))["numOperators"]
preparedBoundaryHeter = prepareLinearSystem(numericalBoundaryHeter)
surfaceWholeHeter = numericalVolumeHeter.numericalOperators.left.geometry.freeSurfaceBoundary.points
    preparedOPTHeter = overlapBoundaryLinearSystem(
    preparedVolumeHeter, preparedBoundaryHeter, surfaceWholeHeter;
    mode=:additive, boundary_weight=1.0,
    )
    preparedConvFDHeter = overlapBoundaryLinearSystem(
        preparedConvFDVolumeHeter, preparedBoundaryHeter, surfaceWholeHeter;
        mode=:additive, boundary_weight=1.0,
    )
    preparedOPT4Heter = runOPT4 ? overlapBoundaryLinearSystem(
        preparedOPT4VolumeHeter, preparedBoundaryHeter, surfaceWholeHeter;
        mode=:additive, boundary_weight=1.0) : nothing
    preparedOPT5Heter = runOPT5 ? overlapBoundaryLinearSystem(
        preparedOPT5VolumeHeter, preparedBoundaryHeter, surfaceWholeHeter;
        mode=:additive, boundary_weight=1.0) : nothing
else
    preparedOPTHeter = preparedVolumeHeter
    preparedConvFDHeter = preparedConvFDVolumeHeter
    preparedOPT4Heter = runOPT4 ? preparedOPT4VolumeHeter : nothing
    preparedOPT5Heter = runOPT5 ? preparedOPT5VolumeHeter : nothing
end
@assert preparedOPTHeter.NForceField == 2
@assert preparedConvFDHeter.NForceField == 2
runOPT4 && @assert preparedOPT4Heter.NForceField == 2
runOPT5 && @assert preparedOPT5Heter.NForceField == 2
optOperatorUnitConventionHeter = optFormulationHeter
boundaryOverlapRowsHeter = hasproperty(preparedOPTHeter, :boundary_overlap_rows) ?
    preparedOPTHeter.boundary_overlap_rows : Int[]
@show quickRun optFormulationHeter optSpatialRefinement dxOPTHeter dtOPTHeter recipeSpacingHeter
@show operatorTimingsHeter
@show preparedOPTHeter.spaceShape preparedConvFDHeter.spaceShape runOPT5 length(boundaryOverlapRowsHeter)
runOPT5 && @show preparedOPT5Heter.spaceShape


In [ ]:
optPaddingHeter = isnothing(bcOPTHeter.cerjan) ? zeros(Int, 2, 2) : cerjan_padding(bcOPTHeter.cerjan)
@assert optOperatorUnitConventionHeter === optFormulationHeter "restart the kernel and rerun the OPT operator cell"
xOPTHeter = range(first(xOPTPhysicalHeter) - optPaddingHeter[1,1] * dxOPTHeter;
    step=dxOPTHeter, length=preparedOPTHeter.spaceShape[1])
zOPTHeter = range(first(zOPTPhysicalHeter) - optPaddingHeter[1,2] * dxOPTHeter;
    step=dxOPTHeter, length=preparedOPTHeter.spaceShape[2])
sourceIXOPTHeter = argmin(abs.(xOPTPhysicalHeter .- sourcePosition.x))
sourceIZOPTHeter = argmin(abs.(zOPTPhysicalHeter .- sourcePosition.z))
sourceIndexOPTHeter = CartesianIndex(
    sourceIXOPTHeter + optPaddingHeter[1,1], sourceIZOPTHeter + optPaddingHeter[1,2])
sourceLinearOPTHeter = LinearIndices(preparedOPTHeter.spaceShape)[sourceIndexOPTHeter]
sourceSpecsOPTHeter = spatial_source_specs(xOPTPhysicalHeter, zOPTPhysicalHeter)
sourceNodesOPTHeter = [(linear=LinearIndices(preparedOPTHeter.spaceShape)[
        CartesianIndex(argmin(abs.(xOPTHeter .- s.x)),
            argmin(abs.(zOPTHeter .- s.z)))], weight=s.weight)
    for s in sourceSpecsOPTHeter]
@assert isapprox(sum(s.weight for s in sourceNodesOPTHeter), 1.0; atol=1e-14)
optStepsHeter = ceil(Int, duration / dtOPTHeter)
optOutputStrideHeter = max(1, round(Int, outputSampling / dtOPTHeter))
# Align the force stencil with (past..., future): for three time points,
# the first solve for u(+Δt) receives f(-Δt), f(0), f(+Δt).
nPastForceLevelsHeter = preparedOPTHeter.timePointsUsedForOneStep - 1
sourceTimesOPTHeter = ((1 - nPastForceLevelsHeter):optStepsHeter) .* dtOPTHeter
@assert sourceTimesOPTHeter[preparedOPTHeter.timePointsUsedForOneStep] == dtOPTHeter
waveletOPTHeter = rickerSource.(sourceTimesOPTHeter)
# In SI, famousEquations uses f=ρa and the cell body-force density is
# sourceForce/(ΔxΔz). Under the scaled-coordinate PDE this becomes
# f'=f Δt²/ρ = sourceForce Δt²/(ρ ΔxΔz).
sourceScaleOPTHeter = optFormulationHeter === :scaled_coordinates ?
    sourceForce * dtOPTHeter^2 / (rho0 * dxOPTHeter^2) :
    sourceForce / dxOPTHeter^2
@assert optFormulationHeter !== :scaled_coordinates ||
    isapprox(sourceScaleOPTHeter * rho0 * dxOPTHeter^2 / dtOPTHeter^2, sourceForce)
sourceOPTHeter = zeros(Float64, preparedOPTHeter.NforcePoints,
    preparedOPTHeter.NForceField, length(sourceTimesOPTHeter))
for sourceNode in sourceNodesOPTHeter
    sourceOPTHeter[sourceNode.linear, 2, :] .+=
        sourceNode.weight .* sourceScaleOPTHeter .* waveletOPTHeter
end
predictorSelectionHeter = optPredictorMode === :lumped_future ?
    (predictor_mode=:lumped_future,) :
    (predictor_mode=:prepared, predictor_prepared=preparedPredictorVolume)
optMarchingOptionsHeter = optTimeMarchingScheme === :direct ?
    (scheme=:direct,) : merge((
        scheme=:predictor_corrector,
        corrector_iterations=optCorrectorIterations,
        corrector_tolerance=optCorrectorTolerance,
        corrector_relaxation=optCorrectorRelaxation,
    ), predictorSelectionHeter)
propagationOPTHeter = propagateLinearSystem(
    preparedOPTHeter, optStepsHeter, dtOPTHeter;
    sourceFull=sourceOPTHeter, output_stride=optOutputStrideHeter,
    blowup_limit=1e8,
    solver_name="OPT3",
    optMarchingOptionsHeter...,
)
@assert !propagationOPTHeter.stopped_early
if !isnothing(propagationOPTHeter.predictor_corrector)
    pcHeter = propagationOPTHeter.predictor_corrector
    @show extrema(pcHeter.iterations_per_step) extrema(pcHeter.relative_residuals)
    @show propagationOPTHeter.timing
end

# Controlled Γ diagnostic. Preserve, for every temporal source slot, the
# signed columnHeter sum of R_force but place it only on the collocated equation
# row. Thus only the spatial redistribution is removed.
function collocated_source_operatorHeter(prepared, source_nodes; force_field=2)
    rowsHeter=Int[]; columns=Int[]; values=Float64[]
    nspaceHeter, nfieldHeter = prepared.NpointsSpace, prepared.NField
    for node in source_nodes, time_slot in 1:prepared.timePointsUsedForOneStep
        columnHeter = node.linear + (force_field-1)*nspaceHeter +
            (time_slot-1)*nspaceHeter*prepared.NForceField
        coefficientHeter = sum(@view prepared.R_force[:, columnHeter])
        push!(rowsHeter, force_field + nfieldHeter*(node.linear-1))
        push!(columns, columnHeter); push!(values, coefficientHeter)
    end
    sparse(rowsHeter, columns, values, size(prepared.R_force)...)
end
RForceCollocatedHeter = collocated_source_operatorHeter(preparedOPTHeter, sourceNodesOPTHeter)
preparedOPTWithoutGammaHeter = runOPTWithoutGamma ?
    flexOPT._prepared_with_matrices(preparedOPTHeter, preparedOPTHeter.A_unknown,
        preparedOPTHeter.L_known, RForceCollocatedHeter; source_mode=:collocated) : nothing
propagationOPTWithoutGammaHeter = runOPTWithoutGamma ? propagateLinearSystem(
    preparedOPTWithoutGammaHeter, optStepsHeter, dtOPTHeter; sourceFull=sourceOPTHeter,
    output_stride=optOutputStrideHeter, blowup_limit=1e8,
    solver_name="OPT3 without Γ redistribution",
) : nothing
runOPTWithoutGamma && @assert !propagationOPTWithoutGammaHeter.stopped_early
uxOPTHeter = propagationOPTHeter.history[:, :, 1, :]
uzOPTHeter = propagationOPTHeter.history[:, :, 2, :]
optTimesHeter = propagationOPTHeter.times
uxOPTWithoutGammaHeter = runOPTWithoutGamma ?
    propagationOPTWithoutGammaHeter.history[:, :, 1, :] : nothing
uzOPTWithoutGammaHeter = runOPTWithoutGamma ?
    propagationOPTWithoutGammaHeter.history[:, :, 2, :] : nothing

# Higher-order runs are gated by ElasticLHSOperatorAudit.ipynb.
if runOPT4
    sourceOPT4Heter = zeros(Float64, preparedOPT4Heter.NforcePoints,
        preparedOPT4Heter.NForceField, length(sourceTimesOPTHeter))
    for sourceNode in sourceNodesOPTHeter
        sourceOPT4Heter[sourceNode.linear, 2, :] .+=
            sourceNode.weight .* sourceScaleOPTHeter .* waveletOPTHeter
    end
    propagationOPT4Heter = propagateLinearSystem(preparedOPT4Heter, optStepsHeter, dtOPTHeter;
        sourceFull=sourceOPT4Heter, output_stride=optOutputStrideHeter,
        blowup_limit=1e8, solver_name="OPT4")
    @assert !propagationOPT4Heter.stopped_early
    uxOPT4Heter = propagationOPT4Heter.history[:, :, 1, :]
    uzOPT4Heter = propagationOPT4Heter.history[:, :, 2, :]
    opt4TimesHeter = propagationOPT4Heter.times
else
    propagationOPT4Heter = uxOPT4Heter = uzOPT4Heter = opt4TimesHeter = nothing
end

# OPT5 remains available, but is skipped in the quick benchmark by default.
if runOPT5
    sourceOPT5Heter = zeros(Float64, preparedOPT5Heter.NforcePoints,
        preparedOPT5Heter.NForceField, length(sourceTimesOPTHeter))
    for sourceNode in sourceNodesOPTHeter
        sourceOPT5Heter[sourceNode.linear, 2, :] .+=
            sourceNode.weight .* sourceScaleOPTHeter .* waveletOPTHeter
    end
    propagationOPT5Heter = propagateLinearSystem(
        preparedOPT5Heter, optStepsHeter, dtOPTHeter;
        sourceFull=sourceOPT5Heter, output_stride=optOutputStrideHeter,
        blowup_limit=1e8, solver_name="OPT5",
    )
    @assert !propagationOPT5Heter.stopped_early
    uxOPT5Heter = propagationOPT5Heter.history[:, :, 1, :]
    uzOPT5Heter = propagationOPT5Heter.history[:, :, 2, :]
    opt5TimesHeter = propagationOPT5Heter.times
else
    propagationOPT5Heter = uxOPT5Heter = uzOPT5Heter = opt5TimesHeter = nothing
end

# Run the less stable delta-test conventional recipe last, so an eventual
# late stop cannot discard the expensive OPT5 result from this session.
sourceConvFDHeter = zeros(Float64, preparedConvFDHeter.NforcePoints,
    preparedConvFDHeter.NForceField, length(sourceTimesOPTHeter))
for sourceNode in sourceNodesOPTHeter
    sourceConvFDHeter[sourceNode.linear, 2, :] .+=
        sourceNode.weight .* sourceScaleOPTHeter .* waveletOPTHeter
end
# orderB=-1 is an interior conventional stencil, not a stable closure for
# the abruptly truncated outer rowsHeter. Stop before the earliest source tail
# can excite those rowsHeter unless the user explicitly requests the diagnostic.
convFDBoundaryGuardHeter = 0.5 / sourceFrequency
convFDBoundarySafeTimeHeter = max(0.0, earliestPEdgeArrivalHeter - convFDBoundaryGuardHeter)
convFDDurationHeter = runConvFDBeyondBoundarySafeTime ? duration :
    min(duration, convFDBoundarySafeTimeHeter)
convFDStepsHeter = ceil(Int, convFDDurationHeter / dtOPTHeter)
propagationConvFDHeter = propagateLinearSystem(
    preparedConvFDHeter, convFDStepsHeter, dtOPTHeter;
    sourceFull=sourceConvFDHeter, output_stride=optOutputStrideHeter,
    blowup_limit=1e8,
    solver_name="convFD3 (orderB=-1)",
)
uxConvFDHeter = propagationConvFDHeter.history[:, :, 1, :]
uzConvFDHeter = propagationConvFDHeter.history[:, :, 2, :]
convFDTimesHeter = propagationConvFDHeter.times
propagationConvFDHeter.stopped_early && @warn(
    "convFD3 stopped early; its partial history remains available",
    final_time=last(convFDTimesHeter), requested_duration=duration)
@show convFDBoundarySafeTimeHeter convFDDurationHeter propagationConvFDHeter.stopped_early
@show dtOPTHeter size(uzOPTHeter) maximum(abs, uxOPTHeter) maximum(abs, uzOPTHeter)
@show size(uzConvFDHeter) maximum(abs, uxConvFDHeter) maximum(abs, uzConvFDHeter)
runOPT5 && @show size(uzOPT5Heter) maximum(abs, uxOPT5Heter) maximum(abs, uzOPT5Heter)



In [ ]:
fdConfigHeter = ElasticThreePointConfig2D(
    pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, cfl=0.38,
)
fdHeter = prepare_elastic_wave_2d(
    modelHeter, (dx, dx);
    material_mask=material,
    boundary_conditions=bcFD,
    config=fdConfigHeter,
)
fdCoordinatesHeter = elastic_wave_coordinates(x, z, fdHeter)
fdPaddingHeter = fdHeter.padding
sourcePhysicalHeter = CartesianIndex(
    argmin(abs.(x .- sourcePosition.x)),
    argmin(abs.(z .- sourcePosition.z)),
)
sourceFDHeter = sourcePhysicalHeter + CartesianIndex(Tuple(fdPaddingHeter[1, :]))
sourceSpecsFDHeter = spatial_source_specs(fdCoordinatesHeter.x, fdCoordinatesHeter.z)
sourceNodesFDHeter = [(index=CartesianIndex(
    argmin(abs.(fdCoordinatesHeter.x .- s.x)),
    argmin(abs.(fdCoordinatesHeter.z .- s.z))), weight=s.weight)
    for s in sourceSpecsFDHeter]
fdStepsHeter = ceil(Int, duration / fdHeter.dt)
fdOutputStrideHeter = max(1, round(Int, outputSampling / fdHeter.dt))
fdFramesXHeter = Matrix{Float32}[copy(fdHeter.ux)]
fdFramesZHeter = Matrix{Float32}[copy(fdHeter.uz)]
fdTimesHeter = Float64[0.0]
for step in 1:fdStepsHeter
    step_elastic_wave_2d!(fdHeter)
    for sourceNode in sourceNodesFDHeter
        elasticWave2D.add_ricker_source!(
            fdHeter, sourceNode.index;
            f0=sourceFrequency, t0=sourceDelay,
            amplitude=sourceForce * sourceNode.weight,
            component=:z, source_kind=:force,
        )
    end
    if step % fdOutputStrideHeter == 0 || step == fdStepsHeter
        push!(fdFramesXHeter, copy(fdHeter.ux))
        push!(fdFramesZHeter, copy(fdHeter.uz)); push!(fdTimesHeter, fdHeter.time)
    end
end
uxFDHeter = cat(fdFramesXHeter...; dims=3)
uzFDHeter = cat(fdFramesZHeter...; dims=3)
@show fdHeter.dt size(uzFDHeter) maximum(abs, uxFDHeter) maximum(abs, uzFDHeter)



In [ ]:
waveformQuantity = @isdefined(waveformQuantity) ? waveformQuantity : :velocity
function displacement_tracesHeter(history, times, xaxis, zindex, receivers)
    tracesHeter = Matrix{Float64}(undef, length(times), length(receivers))
    for (j, receiverHeter) in enumerate(receivers)
        ixHeter = argmin(abs.(xaxis .- receiverHeter))
        tracesHeter[:, j] .= Float64.(history[ixHeter, zindex, :])
    end
    return (time=Float64.(times), values=tracesHeter)
end

function velocity_tracesHeter(history, times, xaxis, zindex, receivers)
    tracesHeter = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiverHeter) in enumerate(receivers)
        ixHeter = argmin(abs.(xaxis .- receiverHeter))
        tracesHeter[:, j] .= diff(Float64.(history[ixHeter, zindex, :])) ./ diff(times)
    end
    return (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=tracesHeter)
end
function velocity_traces_at_pointsHeter(history, times, xaxis, zaxis, receivers;
    sampling=:nearest)
    sampling === :nearest || error("only :nearest is enabled in this cheap benchmark path")
    tracesHeter = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiverHeter) in enumerate(receivers)
        ixHeter = argmin(abs.(xaxis .- receiverHeter.x))
        izHeter = argmin(abs.(zaxis .- receiverHeter.z))
        tracesHeter[:, j] .= diff(Float64.(history[ixHeter, izHeter, :])) ./ diff(times)
    end
    (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=tracesHeter,
        receivers=collect(receivers), sampling)
end
fdReceiverZIndexHeter = argmin(abs.(fdCoordinatesHeter.z .- receiverZ))
optReceiverZIndexHeter = argmin(abs.(zOPTHeter .- receiverZ))
fdDisplacementXHeter = displacement_tracesHeter(uxFDHeter, fdTimesHeter, fdCoordinatesHeter.x,
    fdReceiverZIndexHeter, receiverX)
fdDisplacementZHeter = displacement_tracesHeter(uzFDHeter, fdTimesHeter, fdCoordinatesHeter.x,
    fdReceiverZIndexHeter, receiverX)
optDisplacementXHeter = displacement_tracesHeter(uxOPTHeter, optTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
optDisplacementZHeter = displacement_tracesHeter(uzOPTHeter, optTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
convFDDisplacementXHeter = displacement_tracesHeter(uxConvFDHeter, convFDTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
convFDDisplacementZHeter = displacement_tracesHeter(uzConvFDHeter, convFDTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
optNoGammaDisplacementXHeter = runOPTWithoutGamma ? displacement_tracesHeter(
    uxOPTWithoutGammaHeter, optTimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
optNoGammaDisplacementZHeter = runOPTWithoutGamma ? displacement_tracesHeter(
    uzOPTWithoutGammaHeter, optTimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
opt4DisplacementXHeter = runOPT4 ? displacement_tracesHeter(
    uxOPT4Heter, opt4TimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
opt4DisplacementZHeter = runOPT4 ? displacement_tracesHeter(
    uzOPT4Heter, opt4TimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
opt5DisplacementXHeter = runOPT5 ? displacement_tracesHeter(
    uxOPT5Heter, opt5TimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
opt5DisplacementZHeter = runOPT5 ? displacement_tracesHeter(
    uzOPT5Heter, opt5TimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing

fdVelocityXHeter = velocity_tracesHeter(uxFDHeter, fdTimesHeter, fdCoordinatesHeter.x,
    fdReceiverZIndexHeter, receiverX)
fdVelocityZHeter = velocity_tracesHeter(uzFDHeter, fdTimesHeter, fdCoordinatesHeter.x,
    fdReceiverZIndexHeter, receiverX)
optVelocityXHeter = velocity_tracesHeter(uxOPTHeter, optTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
optVelocityZHeter = velocity_tracesHeter(uzOPTHeter, optTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
convFDVelocityXHeter = velocity_tracesHeter(uxConvFDHeter, convFDTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
convFDVelocityZHeter = velocity_tracesHeter(uzConvFDHeter, convFDTimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX)
optNoGammaVelocityXHeter = runOPTWithoutGamma ? velocity_tracesHeter(
    uxOPTWithoutGammaHeter, optTimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
optNoGammaVelocityZHeter = runOPTWithoutGamma ? velocity_tracesHeter(
    uzOPTWithoutGammaHeter, optTimesHeter, xOPTHeter, optReceiverZIndexHeter, receiverX) : nothing
opt4VelocityXHeter = runOPT4 ? velocity_tracesHeter(uxOPT4Heter, opt4TimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX) : nothing
opt4VelocityZHeter = runOPT4 ? velocity_tracesHeter(uzOPT4Heter, opt4TimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX) : nothing
opt5VelocityXHeter = runOPT5 ? velocity_tracesHeter(uxOPT5Heter, opt5TimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX) : nothing
opt5VelocityZHeter = runOPT5 ? velocity_tracesHeter(uzOPT5Heter, opt5TimesHeter, xOPTHeter,
    optReceiverZIndexHeter, receiverX) : nothing
if waveformQuantity === :velocity
    fdWaveformXHeter, fdWaveformZHeter = fdVelocityXHeter, fdVelocityZHeter
    optWaveformXHeter, optWaveformZHeter = optVelocityXHeter, optVelocityZHeter
    convFDWaveformXHeter, convFDWaveformZHeter = convFDVelocityXHeter, convFDVelocityZHeter
    optNoGammaWaveformXHeter, optNoGammaWaveformZHeter =
        optNoGammaVelocityXHeter, optNoGammaVelocityZHeter
    opt4WaveformXHeter, opt4WaveformZHeter = opt4VelocityXHeter, opt4VelocityZHeter
    opt5WaveformXHeter, opt5WaveformZHeter = opt5VelocityXHeter, opt5VelocityZHeter
    waveformUnitLabelHeter, waveformComponentPrefixHeter = "m/s", "v"
else
    fdWaveformXHeter, fdWaveformZHeter = fdDisplacementXHeter, fdDisplacementZHeter
    optWaveformXHeter, optWaveformZHeter = optDisplacementXHeter, optDisplacementZHeter
    convFDWaveformXHeter, convFDWaveformZHeter = convFDDisplacementXHeter, convFDDisplacementZHeter
    optNoGammaWaveformXHeter, optNoGammaWaveformZHeter =
        optNoGammaDisplacementXHeter, optNoGammaDisplacementZHeter
    opt4WaveformXHeter, opt4WaveformZHeter = opt4DisplacementXHeter, opt4DisplacementZHeter
    opt5WaveformXHeter, opt5WaveformZHeter = opt5DisplacementXHeter, opt5DisplacementZHeter
    waveformUnitLabelHeter, waveformComponentPrefixHeter = "m", "u"
end
fdVelocityHeter, optVelocityHeter = fdVelocityZHeter, optVelocityZHeter # physical velocity aliases
receiverHeter = cld(length(receiverX), 2)
fdTraceHeter = (time=fdWaveformZHeter.time, values=fdWaveformZHeter.values[:, receiverHeter])
optTraceHeter = (time=optWaveformZHeter.time, values=optWaveformZHeter.values[:, receiverHeter])
convFDTraceHeter = (time=convFDWaveformZHeter.time,
    values=convFDWaveformZHeter.values[:, receiverHeter])
optNoGammaTraceHeter = runOPTWithoutGamma ? (time=optNoGammaWaveformZHeter.time,
    values=optNoGammaWaveformZHeter.values[:, receiverHeter]) : nothing
opt4TraceHeter = runOPT4 ? (time=opt4WaveformZHeter.time,
    values=opt4WaveformZHeter.values[:, receiverHeter]) : nothing
opt5TraceHeter = runOPT5 ? (time=opt5WaveformZHeter.time,
    values=opt5WaveformZHeter.values[:, receiverHeter]) : nothing
preSPECFEMTracesHeter = (explicitFD3=fdTraceHeter,
    convFD3=convFDTraceHeter, OPT3=optTraceHeter)
runOPTWithoutGamma && (preSPECFEMTracesHeter = merge(preSPECFEMTracesHeter,
    (OPT3_no_Gamma=optNoGammaTraceHeter,)))
runOPT4 && (preSPECFEMTracesHeter = merge(preSPECFEMTracesHeter, (OPT4=opt4TraceHeter,)))
runOPT5 && (preSPECFEMTracesHeter = merge(preSPECFEMTracesHeter, (OPT5=opt5TraceHeter,)))
preSPECFEMHeter = plot_solver_benchmark(preSPECFEMTracesHeter;
    title="Heterogeneous interior modelHeter — normalized $(waveformComponentPrefixHeter)_z")
display(preSPECFEMHeter.figure)
nothing



In [ ]:
dtSPECFEMHeter = dtOPTHeter
caseDirectoryHeter = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "heterogeneous_kirishima_no_free_surface")
solidZHeter = applyFreeSurface ? findall(z .<= 0.0) : collect(eachindex(z))
zSPECFEMHeter = z[solidZHeter]
vpHeter = vpFieldHeter[:, solidZHeter]
vsHeter = vsFieldHeter[:, solidZHeter]
rhoHeter = rhoFieldHeter[:, solidZHeter]
# Refining FD/OPT must not silently refine the SPECFEM spectral mesh.
specfemNxElementsHeter = round(Int, (last(x)-first(x)) / (4referenceDx))
specfemNzElementsHeter = round(Int, (last(zSPECFEMHeter)-first(zSPECFEMHeter)) / (4referenceDx))
specfemCaseHeter = prepare_specfem2d_case(
    caseDirectoryHeter, x, zSPECFEMHeter, vpHeter, vsHeter, rhoHeter, fill(last(zSPECFEMHeter), nx);
    source=sourcePosition, sources=sourceSpecsReference, receivers=receiverX,
    duration=duration, dt=dtSPECFEMHeter, f0=sourceFrequency,
    source_factor=sourceForce, source_angle=0.0,
    source_time_function=rickerSource,
    nx_elements=specfemNxElementsHeter, nz_elements=specfemNzElementsHeter,
    receiver_z=receiverZ, free_surface=applyFreeSurface,
    snapshot_interval_steps=max(1, round(Int, outputSampling / dtSPECFEMHeter)),
    snapshot_image_type=5, # vertical velocity, matching the seismograms
    output_wavefield_dumps=true,
    wavefield_dump_type=1, # displacementHeter vector (uₓ, u_z)
    binary_wavefield_dumps=true,
)
if runSPECFEM2D
    specfemRunHeter = run_specfem2d_case(specfemCaseHeter.case_directory)
else
    specfemRunHeter = (output=specfemCaseHeter.output,)
end
verticalFilesHeter = find_specfem2d_traces(specfemRunHeter.output; component=:z)
@assert length(verticalFilesHeter) == length(receiverX)
specfemTracesZHeter = [read_specfem2d_trace(file;
    time_shift=specfemCaseHeter.time_axis_shift) for file in verticalFilesHeter]
horizontalFilesHeter = find_specfem2d_traces(specfemRunHeter.output; component=:x)
@assert length(horizontalFilesHeter) == length(receiverX)
specfemTracesXHeter = [read_specfem2d_trace(file;
    time_shift=specfemCaseHeter.time_axis_shift) for file in horizontalFilesHeter]
function integrate_velocity_traceHeter(trace)
    displacementHeter = zeros(Float64, length(trace.values))
    displacementHeter[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .*
        diff(trace.time))
    (time=Float64.(trace.time), values=displacementHeter)
end
specfemWaveformsXHeter = waveformQuantity === :velocity ? specfemTracesXHeter :
    integrate_velocity_traceHeter.(specfemTracesXHeter)
specfemWaveformsZHeter = waveformQuantity === :velocity ? specfemTracesZHeter :
    integrate_velocity_traceHeter.(specfemTracesZHeter)
specfemTracesHeter = specfemWaveformsZHeter # selected-quantity compatibility alias
specfemTraceHeter = specfemWaveformsZHeter[receiverHeter]
specfemWavefieldHeter = read_specfem2d_wavefield_dumps(
    specfemRunHeter.output; dt=dtSPECFEMHeter,
    time_axis_shift=specfemCaseHeter.time_axis_shift,
)
specfemSnapshotVideoHeter = make_specfem2d_snapshot_video(
    specfemRunHeter.output;
    output_path=joinpath(specfemRunHeter.output, "SPECFEM2D_vertical_velocity.mp4"),
    framerate=20,
)
solverLogTextHeter = read(joinpath(specfemCaseHeter.case_directory, "solver.log"), String)
specfemCFLMatchHeter = match(r"Max CFL stability condition[^=]*=\s*([0-9.Ee+-]+)",
    solverLogTextHeter)
specfemCFLHeter = isnothing(specfemCFLMatchHeter) ? missing :
    parse(Float64, specfemCFLMatchHeter.captures[1])
cflSummaryHeter = (
    FD3_axis=vpMaximumHeter * Float64(fdHeter.dt) / dx,
    FD3_2D=sqrt(2) * vpMaximumHeter * Float64(fdHeter.dt) / dx,
    OPT3_axis=vpMaximumHeter * dtOPTHeter / dxOPTHeter,
    OPT3_2D=sqrt(2) * vpMaximumHeter * dtOPTHeter / dxOPTHeter,
    SPECFEM2D_reported=specfemCFLHeter,
)
# SPECFEM uses 5 GLL points per element in this build. Its default number
# of elements makes the mean GLL-node interval comparable to dx.
specfemElementSizeHeter = (
    x=(last(x)-first(x)) / specfemCaseHeter.nx_elements,
    z=(last(zSPECFEMHeter)-first(zSPECFEMHeter)) / specfemCaseHeter.nz_elements,
)
discretizationSummaryHeter = (
    OPT3=(dx=dxOPTHeter, dt=dtOPTHeter),
    SPECFEM2D=(element_size=specfemElementSizeHeter,
        mean_GLL_interval=(x=specfemElementSizeHeter.x/4,
                           z=specfemElementSizeHeter.z/4), dt=dtSPECFEMHeter),
)
@show cflSummaryHeter discretizationSummaryHeter specfemCaseHeter.time_axis_shift
@show specfemCaseHeter.case_directory dtSPECFEMHeter specfemSnapshotVideoHeter.path
if endswith(lowercase(specfemSnapshotVideoHeter.path), ".gif")
    gifDataHeter = base64encode(read(specfemSnapshotVideoHeter.path))
    display(MIME("text/html"),
        "<img src='data:image/gif;base64,$(gifDataHeter)' style='max-width:100%'>")
else
    # VS Code/IJulia renders the MP4 with native playback controls.
    videoURLHeter = "file://" * specfemSnapshotVideoHeter.path
    display(MIME("text/html"),
        "<video controls loop src='$(videoURLHeter)' style='max-width:100%'></video>")
end
specfemSnapshotVideoHeter.path



In [ ]:
function compact_waveform_metricsHeter(referenceHeter, candidate)
    fullHeter = waveform_metrics(referenceHeter, candidate; samples=1001)
    (correlation=fullHeter.correlation, relative_error=fullHeter.relative_error,
     relative_rmse=fullHeter.relative_rmse,
     optimal_amplitude=fullHeter.optimal_amplitude,
     bias=fullHeter.bias, error_variance=fullHeter.error_variance)
end

function solver_traces_for_stationHeter(station)
    tracesHeter = (
        FD3=(time=fdWaveformZHeter.time, values=fdWaveformZHeter.values[:, station]),
        convFD3=(time=convFDWaveformZHeter.time,
            values=convFDWaveformZHeter.values[:, station]),
        OPT3=(time=optWaveformZHeter.time, values=optWaveformZHeter.values[:, station]),
    )
    runOPT4 && (tracesHeter = merge(tracesHeter, (OPT4=(time=opt4WaveformZHeter.time,
        values=opt4WaveformZHeter.values[:, station]),)))
    runOPT5 && (tracesHeter = merge(tracesHeter, (OPT5=(time=opt5WaveformZHeter.time,
        values=opt5WaveformZHeter.values[:, station]),)))
    tracesHeter
end

stationErrorMetricsHeter = map(eachindex(receiverX)) do station
    referenceHeter = specfemWaveformsZHeter[station]
    candidatesHeter = solver_traces_for_stationHeter(station)
    metricsHeter = map(trace -> compact_waveform_metricsHeter(referenceHeter, trace), candidatesHeter)
    (station=station, x_km=receiverX[station]/1e3, metrics=metricsHeter)
end
foreach(display, stationErrorMetricsHeter)

methodNamesHeter = keys(first(stationErrorMetricsHeter).metrics)
globalErrorMetricsHeter = NamedTuple{methodNamesHeter}(map(methodNamesHeter) do method
    rowsHeter = [getproperty(item.metrics, method) for item in stationErrorMetricsHeter]
    (mean_correlation=mean(getproperty.(rowsHeter, :correlation)),
     rms_relative_error=sqrt(mean(abs2, getproperty.(rowsHeter, :relative_error))),
     maximum_relative_error=maximum(getproperty.(rowsHeter, :relative_error)),
     mean_raw_relative_rmse=mean(getproperty.(rowsHeter, :relative_rmse)))
end)
display(globalErrorMetricsHeter)

errorFigureHeter = Figure(size=(1050, 430))
correlationAxisHeter = Axis(errorFigureHeter[1, 1]; xlabel="receiverHeter x (km)",
    ylabel="correlation with SPECFEM2D")
relativeAxisHeter = Axis(errorFigureHeter[1, 2]; xlabel="receiverHeter x (km)",
    ylabel="shape-relative error")
for method in methodNamesHeter
    rowsHeter = [getproperty(item.metrics, method) for item in stationErrorMetricsHeter]
    xsHeter = getproperty.(stationErrorMetricsHeter, :x_km)
    scatterlines!(correlationAxisHeter, xsHeter, getproperty.(rowsHeter, :correlation);
        marker=:circle, label=string(method))
    scatterlines!(relativeAxisHeter, xsHeter, getproperty.(rowsHeter, :relative_error);
        marker=:circle, label=string(method))
end
axislegend(correlationAxisHeter); axislegend(relativeAxisHeter)
errorFigureHeter

